# Modelling Nocturnal Thermal Comfort of Sleepwear in Hot-Humid Climates

Staged run. **Stage 1** downloads and inspects both datasets (reviewed). **Stage 2** filters and cleans ASHRAE with a full row-count log, builds the manikin clo tables and exploratory figures (reviewed). **Stage 3** adds the PMV/PPD baseline and the models with the original comfort label (a documented negative result). **Stage 4** (this version) uses the warm-side discomfort target and adds SHAP, scenarios and the written reports. If pip asks you to restart the runtime after the first cell, restart and Run all again.

How to run on Colab: *Runtime > Run all* (a CPU runtime is enough). The last cell zips the
final results and downloads `stage4_outputs.zip`. Send that zip back so I can check it matches.

Data (both CC BY 4.0):
- ASHRAE Global Thermal Comfort Database II v2.1, https://zenodo.org/records/4993952
- Heat balance of sleeping sets (Newton thermal manikin), https://zenodo.org/records/20179324

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False   # set True to keep everything in Google Drive so it survives runtime resets

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
elif IN_COLAB:
    BASE = Path("/content")
else:
    BASE = Path(os.environ.get("SLEEPWEAR_BASE", Path.cwd()))

PROJECT = BASE / "sleepwear-comfort"
for d in ["src", "data/raw", "results", "figures", "notebooks"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
(PROJECT / ".gitignore").write_text("data/raw/\n__pycache__/\n*.zip\n.ipynb_checkpoints/\n")
print("Project folder:", PROJECT)

In [ ]:
%pip install -q openpyxl requests pythermalcomfort==4.8.1 xgboost==3.4.1 shap==0.54.1 scikit-learn

## Source files
These cells write `src/` so the notebook is self-contained.

In [ ]:
%%writefile src/config.py
"""Project-wide configuration. The ONLY place column names are mapped.

Stage 1 leaves COLUMN_MAP empty on purpose: real column names must be read from
the inspection output first (see results/inspection/), then filled in here.
"""
from __future__ import annotations

import os
from pathlib import Path

SEED = 42

ROOT = Path(__file__).resolve().parents[1]
DATA_RAW = ROOT / "data" / "raw"
RESULTS = ROOT / "results"
INSPECTION = RESULTS / "inspection"
FIGURES = ROOT / "figures"

# Base URL is overridable only so the download code can be tested offline.
ZENODO_BASE = os.environ.get("ZENODO_BASE", "https://zenodo.org")

DATASETS = {
    "ashrae": {
        "record_id": "4993952",
        "title": "ASHRAE Global Thermal Comfort Database II v2.1",
        "license": "CC BY 4.0",
        "url": "https://zenodo.org/records/4993952",
    },
    "manikin": {
        "record_id": "20179324",
        "title": "Heat balance of sleeping sets (Newton thermal manikin)",
        "license": "CC BY 4.0",
        "url": "https://zenodo.org/records/20179324",
    },
}

# ---------------------------------------------------------------------------
# Column mapping: the ONLY place real column names appear. Confirmed against the
# stage 1 inspection output (db_measurements_v2.1.0.csv.gz and db_metadata.csv).
# ---------------------------------------------------------------------------
COLUMN_MAP: dict[str, dict[str, str | None]] = {
    "ashrae": {
        # row-level file
        "record_id": "record_id",
        "building_id": "building_id",       # also the GroupKFold group (decision 4)
        "thermal_sensation": "thermal_sensation",
        "thermal_acceptability": "thermal_acceptability",  # sensitivity check only
        "clo": "clo",
        "air_temp": "ta",
        "rel_humidity": "rh",
        "air_speed": "vel",
        "met": "met",
        "mean_radiant_temp": "tr",          # 34.8% missing, used later for PMV only
        "fan": "fan",                       # 85.3% missing, descriptive only
        # building-level metadata file (joined on building_id)
        "building_type": "building_type",
        "climate": "climate",
        "quality_assurance": "quality_assurance",
        "contributor": "contributor",
        "country": "country",
        "city": "city",
    },
    "manikin": {
        # parsed by label, not by column position (see src/manikin.py)
        "files": {
            "supine_table": "Dataset_Tab 1.xlsx",
            "side_table": "Dataset_Tab 2.xlsx",
            "both_positions": "Dataset_Figure 4.xlsx",
            "torso_segments": "Dataset_Figure 3.xlsx",
        },
    },
}

# ---------------------------------------------------------------------------
# Analysis decisions (confirmed by the user after the stage 1 inspection).
# ---------------------------------------------------------------------------
# Explicit class list (replaces the stage 2 regex, which wrongly matched highland/dry classes and missed
# "wet equatorial"). Matched exactly, case-insensitive, on the ashrae `climate` column.
HOT_HUMID_CLASSES = [
    "tropical wet savanna", "tropical savanna", "tropical monsoon", "tropical", "tropical rainforest",
    "wet equatorial", "humid subtropical", "monsoon-influenced humid subtropical",
]
RESIDENTIAL_TYPES = ["multifamily housing"]       # the only residential class in `building_type`
MET_MAX = 1.3                                     # "low activity" stand-in for rest (sedentary limit)
COMFORT_VOTE_RANGE = (-1, 1)                      # comfortable = rounded sensation vote in [-1, +1]
CLO_PER_M2KW = 1 / 0.155                          # 1 clo = 0.155 m2K/W

# Physical plausibility bounds, applied in order after the completeness filters.
RANGE_RULES = {
    "air_temp": (15.0, 40.0),
    "rel_humidity": (10.0, 100.0),
    "air_speed": (0.0, 2.0),
    "clo": (0.1, 2.5),
}
PROCESSED = ROOT / "data" / "processed"

# Keywords used ONLY to suggest candidate columns in the inspection report.
# They are hints for a human, never used to pick columns automatically.
CONCEPT_KEYWORDS: dict[str, list[str]] = {
    "thermal_sensation": ["sensation", "tsv", "vote"],
    "thermal_comfort": ["comfort", "acceptab"],
    "clo": ["clo", "insulation", "clothing"],
    "air_temp": ["air temp", "ta ", "temperature", "(c)", "tdb"],
    "rel_humidity": ["humid", "rh"],
    "air_speed": ["velocity", "air speed", "airspeed", "v_a"],
    "met": ["met", "metabolic"],
    "building_type": ["building type", "building"],
    "climate": ["climate", "koppen", "köppen"],
    "group_id": ["study", "publication", "contributor", "building id", "dataset", "source", "citation"],
    "set_id": ["set", "garment", "pajama", "pyjama", "sleepwear", "ensemble"],
    "posture": ["posture", "position", "supine", "lateral", "prone"],
    "total_resistance": ["total", "rt", "resistance", "insulation", "clo"],
    "torso_resistance": ["torso", "trunk", "chest", "segment"],
}

# ---------------------------------------------------------------------------
# Modelling decisions (stage 3).
# ---------------------------------------------------------------------------
N_SPLITS = 5                  # StratifiedGroupKFold by building_id
N_BOOT = 500                  # cluster bootstrap replicates (resampling buildings)
COMFORT_THRESHOLD = 0.80      # "stays comfortable" = predicted P(comfortable) >= 0.80 (confirmed)
SLEEP_MET = 0.7               # met used for sleep scenarios (ASHRAE 55 value for sleeping)

# Decisions after the stage 3 review (confirmed by the user):
TARGET_KIND = "warm"          # main target: warm-side discomfort = rounded sensation vote >= +1
WARM_P_MAX = 0.20             # scenario rule: "stays comfortable" = predicted P(warm-side discomfort) <= 0.20
PMV_WARM_CUT = 0.5            # theory-based rule: PMV < +0.5 (rounds to a vote below +1)
N_BOOT_SCEN = 200             # building bootstrap refits for scenario uncertainty

In [ ]:
%%writefile src/download.py
"""Download both Zenodo datasets into data/raw/ (never committed to git).

File names are discovered through the Zenodo record API, not hard-coded.
Each file is checksum-verified when Zenodo supplies an md5. Archives are
extracted next to the download. The record metadata (title, DOI, license) is
saved so the README can credit the datasets correctly.
"""
from __future__ import annotations

import hashlib
import json
import sys
import time
import zipfile
from pathlib import Path

import requests

try:  # works as `python -m src.download` and as plain import in the notebook
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore


def _get(url: str, **kw) -> requests.Response:
    last = None
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=60, **kw)
            r.raise_for_status()
            return r
        except requests.RequestException as e:  # retry with backoff
            last = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to fetch {url}: {last}")


def get_record(record_id: str) -> dict:
    return _get(f"{config.ZENODO_BASE}/api/records/{record_id}").json()


def _file_entries(record: dict) -> list[dict]:
    out = []
    for f in record.get("files", []):
        name = f.get("key") or f.get("filename")
        url = (f.get("links") or {}).get("self") or (f.get("links") or {}).get("download")
        if not url:
            url = f"{config.ZENODO_BASE}/records/{record['id']}/files/{name}?download=1"
        out.append({"name": name, "size": f.get("size"), "checksum": f.get("checksum"), "url": url})
    return out


def _md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _safe_extract(zpath: Path, dest: Path) -> list[str]:
    dest.mkdir(parents=True, exist_ok=True)
    names = []
    with zipfile.ZipFile(zpath) as z:
        for member in z.infolist():
            target = (dest / member.filename).resolve()
            if not str(target).startswith(str(dest.resolve())):
                raise RuntimeError(f"Unsafe path in zip: {member.filename}")
        z.extractall(dest)
        names = z.namelist()
    return names


def download_record(key: str, force: bool = False) -> Path:
    meta = config.DATASETS[key]
    rid = meta["record_id"]
    dest = config.DATA_RAW / key
    dest.mkdir(parents=True, exist_ok=True)

    record = get_record(rid)
    (dest / "record_metadata.json").write_text(json.dumps(record, indent=2))
    md = record.get("metadata", {})
    print(f"[{key}] {md.get('title')}")
    print(f"[{key}] DOI: {record.get('doi') or md.get('doi')} | "
          f"license: {(md.get('license') or {}).get('id')} | "
          f"version: {md.get('version')}")

    for f in _file_entries(record):
        path = dest / f["name"]
        if path.exists() and not force:
            print(f"[{key}] already have {f['name']}")
        else:
            print(f"[{key}] downloading {f['name']} ({f['size']} bytes)")
            with _get(f["url"], stream=True) as r, open(path, "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        if f["checksum"] and f["checksum"].startswith("md5:"):
            ok = _md5(path) == f["checksum"].split(":", 1)[1]
            print(f"[{key}] md5 {'OK' if ok else 'MISMATCH'}: {f['name']}")
            if not ok:
                raise RuntimeError(f"Checksum mismatch for {f['name']}")
        if path.suffix.lower() == ".zip":
            names = _safe_extract(path, dest / "extracted")
            print(f"[{key}] extracted {len(names)} entries into {dest / 'extracted'}")
    return dest


def main() -> None:
    for key in config.DATASETS:
        download_record(key)
    print("Done. Raw data is in", config.DATA_RAW)


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile src/inspect_data.py
"""Inspect every raw table: real column names, dtypes, missing counts, ranges.

Nothing here assumes a column name. Outputs go to results/inspection/ so they
can be reviewed (and sent back) before any cleaning or modelling code is written.
"""
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

TABLE_SUFFIXES = {".csv", ".gz", ".xlsx", ".xls", ".tsv"}
TEXT_SUFFIXES = {".txt", ".md", ".json", ".rst"}
NA_LIKE = {"", "na", "n/a", "nan", "none", "null", "-", "--", "?", "NA"}


def _safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")


def find_files(folder: Path) -> list[Path]:
    skip = {"record_metadata.json"}
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.name not in skip)


def _read_csv(path: Path) -> pd.DataFrame:
    sep = "\t" if ".tsv" in path.name.lower() else ","
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, sep=sep)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Could not decode {path}")


def load_tables(folder: Path) -> dict[str, pd.DataFrame]:
    """Return {label: DataFrame} for every csv/xlsx table under folder."""
    tables: dict[str, pd.DataFrame] = {}
    for p in find_files(folder):
        suf = p.suffix.lower()
        rel = str(p.relative_to(folder))
        if suf in {".csv", ".tsv"} or (suf == ".gz" and p.name.lower().endswith((".csv.gz", ".tsv.gz"))):
            tables[rel] = _read_csv(p)
        elif suf in {".xlsx", ".xls"}:
            xl = pd.ExcelFile(p)
            for sh in xl.sheet_names:
                tables[f"{rel}::{sh}"] = xl.parse(sh)
    return tables


def profile(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        r = {
            "column": col,
            "dtype": str(s.dtype),
            "n_rows": n,
            "n_missing": int(s.isna().sum()),
            "pct_missing": round(100 * s.isna().mean(), 2) if n else np.nan,
            "n_unique": int(s.nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            r.update(min=s.min(), max=s.max(), mean=round(float(s.mean()), 4) if s.notna().any() else np.nan,
                     median=s.median())
        else:
            as_str = s.dropna().astype(str)
            r["na_like_strings"] = int(as_str.str.strip().str.lower().isin(NA_LIKE).sum())
            num = pd.to_numeric(as_str, errors="coerce")
            r["numeric_like_share"] = round(float(num.notna().mean()), 3) if len(as_str) else np.nan
        if r["n_unique"] <= 15 or not pd.api.types.is_numeric_dtype(s):
            vc = s.value_counts(dropna=True).head(6)
            r["top_values"] = "; ".join(f"{k} ({v})" for k, v in vc.items())
        rows.append(r)
    return pd.DataFrame(rows)


def raw_preview(path: Path, nrows: int = 15) -> str:
    """First rows with NO header interpretation: reveals unit rows / multi-row headers."""
    suf = path.suffix.lower()
    try:
        if suf in {".xlsx", ".xls"}:
            xl = pd.ExcelFile(path)
            parts = []
            for sh in xl.sheet_names:
                d = xl.parse(sh, header=None, nrows=nrows)
                parts.append(f"--- sheet '{sh}' (raw, header=None) ---\n{d.to_string(max_cols=20)}")
            return "\n".join(parts)
        if suf in {".csv", ".tsv"}:
            d = pd.read_csv(path, header=None, nrows=nrows, encoding="latin-1", sep=None, engine="python")
            return d.to_string(max_cols=20)
    except Exception as e:  # preview is best effort
        return f"(preview failed: {e})"
    return ""


def text_files(folder: Path, limit: int = 4000) -> dict[str, str]:
    out = {}
    for p in find_files(folder):
        if p.suffix.lower() in TEXT_SUFFIXES:
            out[str(p.relative_to(folder))] = p.read_text(errors="replace")[:limit]
    return out


def suggest_candidates(tables: dict[str, pd.DataFrame], concepts: list[str]) -> pd.DataFrame:
    rows = []
    for concept in concepts:
        kws = [k.lower() for k in config.CONCEPT_KEYWORDS[concept]]
        for label, df in tables.items():
            for col in df.columns:
                c = f"{str(col).lower()} "
                if any(k in c for k in kws):
                    rows.append({"concept": concept, "table": label, "column": col})
    return pd.DataFrame(rows)


def inspect_dataset(key: str) -> dict[str, pd.DataFrame]:
    folder = config.DATA_RAW / key
    out = config.INSPECTION / key
    out.mkdir(parents=True, exist_ok=True)
    pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 80)

    print("=" * 100)
    print(f"DATASET: {key}  |  {config.DATASETS[key]['title']}")
    print("=" * 100)
    files = find_files(folder)
    print("Files on disk:")
    for p in files:
        print(f"  {p.relative_to(folder)}  ({p.stat().st_size:,} bytes)")

    for name, txt in text_files(folder).items():
        print(f"\n--- text file: {name} (first 4000 chars) ---\n{txt}")
        (out / f"textfile__{_safe_name(name)}.txt").write_text(txt)

    for p in files:
        if p.suffix.lower() in {".xlsx", ".xls", ".csv", ".tsv"} and p.stat().st_size < 5_000_000:
            prev = raw_preview(p)
            print(f"\n--- RAW PREVIEW: {p.name} ---\n{prev}")
            (out / f"rawpreview__{_safe_name(p.name)}.txt").write_text(prev)

    tables = load_tables(folder)
    for label, df in tables.items():
        print(f"\n### TABLE {label}: {df.shape[0]:,} rows x {df.shape[1]} columns")
        prof = profile(df)
        print(prof.drop(columns=["n_rows"]).to_string(index=False))
        prof.to_csv(out / f"columns__{_safe_name(label)}.csv", index=False)
        df.head(10).to_csv(out / f"head__{_safe_name(label)}.csv", index=False)
        dups = int(df.duplicated().sum())
        print(f"Fully duplicated rows: {dups:,}")
    return tables


def write_candidates(all_tables: dict[str, dict[str, pd.DataFrame]]) -> pd.DataFrame:
    parts = []
    for key, concepts in (("ashrae", ["thermal_sensation", "thermal_comfort", "clo", "air_temp", "rel_humidity",
                                      "air_speed", "met", "building_type", "climate", "group_id"]),
                          ("manikin", ["set_id", "posture", "total_resistance", "torso_resistance"])):
        cand = suggest_candidates(all_tables.get(key, {}), concepts)
        if len(cand):
            cand.insert(0, "dataset", key)
            parts.append(cand)
    res = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    config.INSPECTION.mkdir(parents=True, exist_ok=True)
    res.to_csv(config.INSPECTION / "candidate_columns.csv", index=False)
    return res


if __name__ == "__main__":
    tabs = {k: inspect_dataset(k) for k in config.DATASETS}
    print(write_candidates(tabs).to_string(index=False))

In [ ]:
%%writefile src/clean.py
"""ASHRAE subsetting and cleaning with a full audit log.

Every filter is applied sequentially and logged (rows before/after, buildings
remaining) to results/filter_log.csv, so the report can state exactly how many
rows each rule removed.
"""
from __future__ import annotations

import pandas as pd

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

M = config.COLUMN_MAP["ashrae"]


def load_ashrae() -> pd.DataFrame:
    folder = config.DATA_RAW / "ashrae"
    meas_path = next(folder.glob("db_measurements*.csv.gz"))
    meta_path = folder / "db_metadata.csv"
    meas = pd.read_csv(meas_path, low_memory=False)
    meta = pd.read_csv(meta_path)
    meta_cols = [M[k] for k in ("building_id", "building_type", "climate", "quality_assurance",
                                "contributor", "country", "city")]
    df = meas.merge(meta[meta_cols], on=M["building_id"], how="left", validate="many_to_one")
    unmatched = int(df[M["building_type"]].isna().sum())
    print(f"Joined metadata on {M['building_id']}: {len(df):,} rows, {unmatched:,} without metadata")
    return df


class FilterLog:
    def __init__(self, df: pd.DataFrame):
        self.rows: list[dict] = []
        self._add("0. raw measurements joined with metadata", "none", len(df), len(df), df)

    def _add(self, step, rule, before, after, df):
        self.rows.append({
            "step": step, "rule": rule, "rows_before": before, "rows_after": after,
            "rows_dropped": before - after,
            "buildings_after": int(df[M["building_id"]].nunique()) if len(df) else 0,
        })

    def apply(self, df: pd.DataFrame, mask: pd.Series, step: str, rule: str) -> pd.DataFrame:
        out = df[mask.fillna(False)]
        self._add(step, rule, len(df), len(out), out)
        return out

    def table(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)


def _is_hot_humid(x) -> bool:
    return isinstance(x, str) and x.strip().lower() in config.HOT_HUMID_CLASSES


def climate_report(df: pd.DataFrame) -> pd.DataFrame:
    """Rows and buildings per climate class, flagged by the hot-humid rule."""
    g = df.groupby(M["climate"]).agg(rows=(M["record_id"], "size"),
                                      buildings=(M["building_id"], "nunique")).reset_index()
    g["matches_hot_humid_rule"] = g[M["climate"]].map(_is_hot_humid)
    return g.sort_values("rows", ascending=False)


def build_subset(df: pd.DataFrame):
    """Return (clean_df, filter_log_table). Order matters and is documented in the log."""
    log = FilterLog(df)
    d = df

    d = log.apply(d, d[M["quality_assurance"]].eq("pass"), "1. quality assurance",
                  "metadata quality_assurance == 'pass'")
    d = log.apply(d, d[M["climate"]].map(_is_hot_humid),
                  "2. hot-humid climate", "climate in explicit hot-humid class list (config.HOT_HUMID_CLASSES)")
    d = log.apply(d, d[M["building_type"]].isin(config.RESIDENTIAL_TYPES), "3. residential",
                  f"building_type in {config.RESIDENTIAL_TYPES}")
    d = log.apply(d, d[M["met"]] <= config.MET_MAX, "4. low metabolic rate",
                  f"met <= {config.MET_MAX} (rows with missing met are dropped)")

    for key in ("thermal_sensation", "air_temp", "rel_humidity", "air_speed", "clo"):
        col = M[key]
        d = log.apply(d, d[col].notna(), f"5. complete cases: {col}", f"{col} is not missing")

    for key, (lo, hi) in config.RANGE_RULES.items():
        col = M[key]
        d = log.apply(d, d[col].between(lo, hi), f"6. plausible range: {col}", f"{lo} <= {col} <= {hi}")

    d = d.copy()
    d["vote_rounded"] = d[M["thermal_sensation"]].round().clip(-3, 3).astype(int)
    lo, hi = config.COMFORT_VOTE_RANGE
    d["comfortable"] = d["vote_rounded"].between(lo, hi).astype(int)
    return d, log.table()


def met_sensitivity(df: pd.DataFrame, cutoffs=(1.0, 1.2, 1.3, 1.5)) -> pd.DataFrame:
    """Rows left after climate+residential+met filters for alternative met cutoffs."""
    base = df[df[M["climate"]].map(_is_hot_humid)
              & df[M["building_type"]].isin(config.RESIDENTIAL_TYPES)
              & df[M["quality_assurance"]].eq("pass")]
    rows = [{"met_max": c, "rows": int((base[M["met"]] <= c).sum())} for c in cutoffs]
    rows.append({"met_max": "any (met not required)", "rows": len(base)})
    return pd.DataFrame(rows)


def save_processed(d: pd.DataFrame) -> str:
    config.PROCESSED.mkdir(parents=True, exist_ok=True)
    keep = [M[k] for k in ("record_id", "building_id", "thermal_sensation", "thermal_acceptability", "clo",
                           "air_temp", "rel_humidity", "air_speed", "met", "mean_radiant_temp", "fan",
                           "building_type", "climate", "contributor", "country", "city")]
    keep += ["vote_rounded", "comfortable"]
    path = config.PROCESSED / "ashrae_clean.csv.gz"
    d[keep].to_csv(path, index=False)
    return str(path)

In [ ]:
%%writefile src/manikin.py
"""Parse the manikin workbooks into tidy tables, by label rather than position.

Dataset: Heat balance of sleeping sets (Newton thermal manikin).
The 'sets' are bedding configurations (Without cover, Cotton cover, Blanket,
Summer duvet), measured on the back (supine) and on the left side. They are NOT
garment types; the README does not describe the garments worn.
"""
from __future__ import annotations

import re

import pandas as pd

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

FILES = config.COLUMN_MAP["manikin"]["files"]
SET_ORDER = ["Without cover", "Cotton cover", "Blanket", "Summer duvet"]


def _find(name: str):
    root = config.DATA_RAW / "manikin"
    hits = list(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def _raw(name: str) -> pd.DataFrame:
    return pd.read_excel(_find(name), header=None)


def _norm_set(label: str) -> str:
    s = re.sub(r"^\s*\d+\)\s*", "", str(label)).strip()
    s = s.replace("duver", "duvet")  # typo present in the source file
    return s[:1].upper() + s[1:]


def parse_table(key: str, posture: str) -> pd.DataFrame:
    """Tab 1 (supine) or Tab 2 (side): replicate and mean resistances per set."""
    raw = _raw(FILES[key])
    labels = raw.iloc[:, 0].fillna("").astype(str).str.strip()
    set_row = next(i for i in range(len(raw)) if raw.iloc[i, 1:].fillna("").astype(str).str.contains("cover|Blanket|duvet", case=False).any())
    set_cols = [c for c in range(1, raw.shape[1]) if pd.notna(raw.iloc[set_row, c])]
    out = []
    for c in set_cols:
        name = _norm_set(raw.iloc[set_row, c])
        for i, lab in enumerate(labels):
            if re.fullmatch(r"Rtot\d*|UR", lab):
                out.append({"set": name, "posture": posture, "stat": lab,
                            "m2K_per_W": float(raw.iloc[i, c]), "clo": float(raw.iloc[i, c + 1])})
    return pd.DataFrame(out)


def parse_figure4() -> pd.DataFrame:
    raw = _raw(FILES["both_positions"])
    rows = []
    for i in range(len(raw)):
        lab = str(raw.iloc[i, 0])
        if re.match(r"^\s*\d+\)", lab):
            rows.append({"set": _norm_set(lab),
                         "side_clo": float(raw.iloc[i, 1]), "side_err": float(raw.iloc[i, 2]),
                         "supine_clo": float(raw.iloc[i, 3]), "supine_err": float(raw.iloc[i, 4])})
    return pd.DataFrame(rows)


def parse_segments() -> pd.DataFrame:
    raw = _raw(FILES["torso_segments"])
    out, posture, cols = [], None, None
    for i in range(len(raw)):
        c1 = str(raw.iloc[i, 1])
        if "Position" in c1:
            posture = "side" if "side" in c1.lower() else "supine"
        if str(raw.iloc[i, 2]) in ("Chest", "Abdomen", "Back"):
            cols = [str(raw.iloc[i, k]) for k in (2, 3, 4)]
        lab = str(raw.iloc[i, 1])
        if re.match(r"^\s*\d+\)", lab) and posture and cols:
            for k, seg in zip((2, 3, 4), cols):
                out.append({"set": _norm_set(lab), "posture": posture, "segment": seg,
                            "clo": float(raw.iloc[i, k])})
    seg = pd.DataFrame(out)
    seg["suspect_artifact"] = seg["clo"] > 5  # >5 clo is not physical for a torso segment
    return seg


def build_clo_table() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (clo_table, replicate_table, segments) and print the cross-checks."""
    reps = pd.concat([parse_table("supine_table", "supine"), parse_table("side_table", "side")],
                     ignore_index=True)
    mean = reps[reps["stat"] == "Rtot"].rename(columns={"clo": "clo_total", "m2K_per_W": "rtot_m2K_per_W"})
    ur = reps[reps["stat"] == "UR"].rename(columns={"clo": "clo_uncertainty"})[["set", "posture", "clo_uncertainty"]]
    tbl = mean.drop(columns="stat").merge(ur, on=["set", "posture"])
    tbl["set"] = pd.Categorical(tbl["set"], SET_ORDER, ordered=True)
    tbl = tbl.sort_values(["posture", "set"]).reset_index(drop=True)

    f4 = parse_figure4()
    chk = []
    for _, r in f4.iterrows():
        for post in ("supine", "side"):
            t = tbl[(tbl["set"] == r["set"]) & (tbl["posture"] == post)]["clo_total"].iloc[0]
            chk.append({"set": r["set"], "posture": post, "table_clo": t, "figure4_clo": r[f"{post}_clo"],
                        "abs_diff": abs(t - r[f"{post}_clo"])})
    chk = pd.DataFrame(chk)
    print("Cross-check Tab 1/2 vs Figure 4 (max abs diff = %.3f clo)" % chk["abs_diff"].max())
    unit = (tbl["rtot_m2K_per_W"] * config.CLO_PER_M2KW - tbl["clo_total"]).abs().max()
    print("Unit check: max |m2K/W / 0.155 - clo| = %.3f clo" % unit)
    return tbl, reps, parse_segments()

In [ ]:
%%writefile src/explore.py
"""Exploratory figures for the cleaned ASHRAE subset and the manikin clo values.

Every figure is saved as PNG (300 dpi) and SVG into figures/.
"""
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

M = config.COLUMN_MAP["ashrae"]
MIN_BIN = 30  # bins with fewer rows are not plotted

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "font.family": "DejaVu Sans",
})
C = {"main": "#1f5f8b", "alt": "#c4572b", "grey": "#6b7280"}


def save(fig, name: str) -> None:
    config.FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(config.FIGURES / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(config.FIGURES / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)


def _binned_share(d: pd.DataFrame, xcol: str, bins) -> pd.DataFrame:
    b = pd.cut(d[xcol], bins)
    g = d.groupby(b, observed=True)["comfortable"].agg(["mean", "size"]).reset_index()
    g["x"] = g[xcol].apply(lambda i: i.mid)
    g["se"] = np.sqrt(g["mean"] * (1 - g["mean"]) / g["size"])
    return g[g["size"] >= MIN_BIN]


def fig_vote_distribution(d):
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    vc = d["vote_rounded"].value_counts().reindex(range(-3, 4), fill_value=0)
    ax.bar(vc.index, vc.values, color=C["main"])
    ax.axvspan(-1.5, 1.5, color=C["alt"], alpha=0.08)
    ax.set_xlabel("Thermal sensation vote (-3 cold ... +3 hot)")
    ax.set_ylabel("Number of votes")
    ax.set_title(f"Votes in hot-humid residential subset (n = {len(d):,})")
    save(fig, "fig01_vote_distribution")


def fig_comfort_vs_clo(d):
    g = _binned_share(d, M["clo"], np.arange(0.1, 1.6, 0.15))
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    ax.errorbar(g["x"], 100 * g["mean"], yerr=100 * 1.96 * g["se"], marker="o", color=C["main"], capsize=3)
    ax.set_xlabel("Clothing insulation (clo)")
    ax.set_ylabel("Votes within -1 to +1 (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Comfort share by clothing insulation (95% CI)")
    save(fig, "fig02_comfort_vs_clo")


def fig_comfort_vs_temp_by_speed(d):
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    for label, mask, col in (("Air speed < 0.2 m/s", d[M["air_speed"]] < 0.2, C["main"]),
                             ("Air speed >= 0.2 m/s", d[M["air_speed"]] >= 0.2, C["alt"])):
        g = _binned_share(d[mask], M["air_temp"], np.arange(18, 38, 1.5))
        if len(g):
            ax.errorbar(g["x"], 100 * g["mean"], yerr=100 * 1.96 * g["se"], marker="o", capsize=3,
                        color=col, label=f"{label} (n = {int(mask.sum()):,})")
    ax.set_xlabel("Air temperature (degC)")
    ax.set_ylabel("Votes within -1 to +1 (%)")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False)
    ax.set_title("Comfort share by air temperature and air speed")
    save(fig, "fig03_comfort_vs_temp_by_airspeed")


def fig_sensation_heatmap(d):
    tb = pd.cut(d[M["air_temp"]], np.arange(20, 38, 2))
    rb = pd.cut(d[M["rel_humidity"]], np.arange(20, 101, 16))
    piv = d.assign(t=tb, r=rb).pivot_table(index="r", columns="t", values=M["thermal_sensation"],
                                           aggfunc="mean", observed=True)
    cnt = d.assign(t=tb, r=rb).pivot_table(index="r", columns="t", values=M["thermal_sensation"],
                                           aggfunc="size", observed=True)
    piv = piv.where(cnt >= MIN_BIN)
    fig, ax = plt.subplots(figsize=(6.2, 3.9))
    im = ax.imshow(piv.values, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set_xticks(range(piv.shape[1]), [f"{i.left:.0f}-{i.right:.0f}" for i in piv.columns])
    ax.set_yticks(range(piv.shape[0]), [f"{i.left:.0f}-{i.right:.0f}" for i in piv.index])
    ax.grid(False)
    ax.set_xlabel("Air temperature (degC)")
    ax.set_ylabel("Relative humidity (%)")
    ax.set_title(f"Mean sensation vote (cells with n >= {MIN_BIN})")
    fig.colorbar(im, ax=ax, label="Mean vote")
    save(fig, "fig04_sensation_heatmap_temp_rh")


def fig_manikin_clo(tbl: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    sets = list(tbl["set"].cat.categories)
    w = 0.38
    for k, (post, col, lab) in enumerate((("supine", C["main"], "On the back (supine)"),
                                          ("side", C["alt"], "On the left side"))):
        t = tbl[tbl["posture"] == post].set_index("set").loc[sets]
        ax.bar(np.arange(len(sets)) + (k - 0.5) * w, t["clo_total"], w, yerr=t["clo_uncertainty"],
               capsize=3, color=col, label=lab)
    ax.set_xticks(range(len(sets)), sets)
    ax.set_ylabel("Total insulation (clo)")
    ax.set_title("Thermal manikin: total insulation of sleeping sets")
    ax.legend(frameon=False)
    save(fig, "fig05_manikin_total_clo")


def fig_clo_coverage(d: pd.DataFrame, tbl: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    ax.hist(d[M["clo"]], bins=np.arange(0.1, 2.6, 0.1), color=C["grey"], alpha=0.8)
    ax.axvspan(tbl["clo_total"].min(), tbl["clo_total"].max(), color=C["alt"], alpha=0.15,
               label="Manikin sleeping-set range")
    ax.set_xlabel("Clothing insulation (clo)")
    ax.set_ylabel("ASHRAE votes in subset")
    ax.set_title("Training-data clo range vs manikin sleeping-set clo")
    ax.legend(frameon=False)
    save(fig, "fig06_clo_range_coverage")


def run_all(d: pd.DataFrame, tbl: pd.DataFrame) -> None:
    fig_vote_distribution(d)
    fig_comfort_vs_clo(d)
    fig_comfort_vs_temp_by_speed(d)
    fig_sensation_heatmap(d)
    fig_manikin_clo(tbl)
    fig_clo_coverage(d, tbl)


# ----------------------------------------------------------------------------- stage 3 figures
def fig_roc(oof: pd.DataFrame) -> None:
    from sklearn.metrics import roc_auc_score, roc_curve
    y = oof["y"].to_numpy()
    series = [("PMV (ISO 7730)", -np.abs(oof["pmv_iso"]), C["grey"]),
              ("PMV (ASHRAE 55, cooling effect)", -np.abs(oof["pmv_ashrae"]), "#9aa3ad"),
              ("Logistic (quadratic)", oof["p_comfort__Logistic regression (quadratic)"], C["main"]),
              ("XGBoost", oof["p_comfort__XGBoost"], C["alt"])]
    fig, ax = plt.subplots(figsize=(5.0, 4.6))
    for name, sc, col in series:
        fpr, tpr, _ = roc_curve(y, sc)
        ax.plot(fpr, tpr, color=col, label=f"{name} (AUC {roc_auc_score(y, sc):.2f})")
    ax.plot([0, 1], [0, 1], "--", color="black", lw=0.8)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Comfort classification, held-out buildings")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    save(fig, "fig07_roc_curves")


def fig_calibration(oof: pd.DataFrame) -> None:
    from sklearn.calibration import calibration_curve
    y = oof["y"].to_numpy()
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    for name, col in (("Logistic (quadratic)", C["main"]), ("XGBoost", C["alt"])):
        key = "p_comfort__Logistic regression (quadratic)" if "Logistic" in name else "p_comfort__XGBoost"
        frac, mean_p = calibration_curve(y, oof[key], n_bins=8, strategy="quantile")
        ax.plot(100 * mean_p, 100 * frac, marker="o", color=col, label=name)
    ax.plot([40, 100], [40, 100], "--", color="black", lw=0.8)
    ax.set_xlabel("Predicted probability comfortable (%)")
    ax.set_ylabel("Observed share comfortable (%)")
    ax.set_title("Calibration on held-out buildings")
    ax.legend(frameon=False)
    save(fig, "fig08_calibration")


def fig_roc_warm(oof: pd.DataFrame) -> None:
    from sklearn.metrics import roc_auc_score, roc_curve
    y = oof["y"].to_numpy()
    series = [("PMV (ISO 7730)", oof["pmv_iso"], C["grey"]),
              ("PMV (ASHRAE 55, cooling effect)", oof["pmv_ashrae"], "#9aa3ad"),
              ("Logistic regression", oof["p_comfort__Logistic regression (linear)"], C["main"]),
              ("XGBoost", oof["p_comfort__XGBoost"], C["alt"])]
    fig, ax = plt.subplots(figsize=(5.0, 4.6))
    for name, sc_, col in series:
        fpr, tpr, _ = roc_curve(y, sc_)
        ax.plot(fpr, tpr, color=col, label=f"{name} (AUC {roc_auc_score(y, sc_):.2f})")
    ax.plot([0, 1], [0, 1], "--", color="black", lw=0.8)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Warm-side discomfort, held-out buildings")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    save(fig, "fig07_roc_curves_warm")


def fig_calibration_warm(oof: pd.DataFrame) -> None:
    from sklearn.calibration import calibration_curve
    y = oof["y"].to_numpy()
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    for name, key, col in (("Logistic regression", "p_comfort__Logistic regression (linear)", C["main"]),
                           ("XGBoost", "p_comfort__XGBoost", C["alt"])):
        frac, mean_p = calibration_curve(y, oof[key], n_bins=8, strategy="quantile")
        ax.plot(100 * mean_p, 100 * frac, marker="o", color=col, label=name)
    ax.plot([0, 70], [0, 70], "--", color="black", lw=0.8)
    ax.set_xlabel("Predicted warm-side discomfort (%)")
    ax.set_ylabel("Observed share with vote >= +1 (%)")
    ax.set_title("Calibration on held-out buildings")
    ax.legend(frameon=False)
    save(fig, "fig08_calibration_warm")

In [ ]:
%%writefile src/models.py
"""PMV/PPD baselines and interpretable ML models with building-grouped validation.

Inputs for every model (same information for fairness): clo, air temperature,
relative humidity, air speed, met. PMV additionally uses mean radiant temperature
(tr), falling back to air temperature where tr is missing.

Validation: StratifiedGroupKFold by building_id (never a random row split), plus a
leave-one-contributor-out check. Metrics are pooled out-of-fold, with 95% cluster
bootstrap intervals (resampling whole buildings).
"""
from __future__ import annotations

import json
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, LeaveOneGroupOut,
                                     StratifiedGroupKFold)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

M = config.COLUMN_MAP["ashrae"]
FEATURES = [M["clo"], M["air_temp"], M["rel_humidity"], M["air_speed"], M["met"]]
SENS = M["thermal_sensation"]
GROUP = M["building_id"]
Y = "y"   # binary target column; set with set_target()


# ----------------------------------------------------------------------------- data
def load_clean() -> pd.DataFrame:
    return pd.read_csv(config.PROCESSED / "ashrae_clean.csv.gz")


def set_target(df: pd.DataFrame, kind: str) -> pd.DataFrame:
    """kind: 'warm' (vote >= +1, MAIN target) or 'comfortable' (vote in [-1,+1], stage 3 negative result)."""
    d = df.copy()
    d[Y] = (d["vote_rounded"] >= 1).astype(int) if kind == "warm" else d["comfortable"].astype(int)
    return d


# ----------------------------------------------------------------------------- PMV baseline
def pmv_baseline(df: pd.DataFrame) -> pd.DataFrame:
    """PMV/PPD with pythermalcomfort: ISO 7730 and ASHRAE 55 (adds the elevated-air-speed cooling effect)."""
    from pythermalcomfort.models import pmv_ppd_ashrae, pmv_ppd_iso
    from pythermalcomfort.utilities import v_relative

    tdb = df[M["air_temp"]].to_numpy(float)
    tr = df[M["mean_radiant_temp"]].fillna(df[M["air_temp"]]).to_numpy(float)
    vr = np.asarray(v_relative(v=df[M["air_speed"]].to_numpy(float), met=df[M["met"]].to_numpy(float)), float)
    kw = dict(tdb=tdb, tr=tr, vr=vr, rh=df[M["rel_humidity"]].to_numpy(float),
              met=df[M["met"]].to_numpy(float), clo=df[M["clo"]].to_numpy(float), limit_inputs=False)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # pythermalcomfort warns per row when the cooling effect is 0
        iso, ash = pmv_ppd_iso(**kw), pmv_ppd_ashrae(**kw)
    out = pd.DataFrame({"pmv_iso": np.asarray(iso.pmv, float), "ppd_iso": np.asarray(iso.ppd, float),
                        "pmv_ashrae": np.asarray(ash.pmv, float), "ppd_ashrae": np.asarray(ash.ppd, float)},
                       index=df.index)
    out.attrs["n_tr_imputed"] = int(df[M["mean_radiant_temp"]].isna().sum())
    return out


# ----------------------------------------------------------------------------- model factories
def _logit(poly: bool):
    steps = [StandardScaler()]
    if poly:
        steps += [PolynomialFeatures(2, include_bias=False), StandardScaler()]
    steps.append(LogisticRegression(max_iter=5000))
    return make_pipeline(*steps), {"logisticregression__C": [0.03, 0.1, 0.3, 1, 3]}


def _ridge(poly: bool):
    steps = [StandardScaler()]
    if poly:
        steps += [PolynomialFeatures(2, include_bias=False), StandardScaler()]
    steps.append(Ridge())
    return make_pipeline(*steps), {"ridge__alpha": [0.1, 1, 10, 100]}


XGB_GRID = {"max_depth": [2, 3, 4], "n_estimators": [100, 250]}
XGB_FIXED = dict(learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                 tree_method="hist", n_jobs=2, random_state=config.SEED, verbosity=0)


def classifiers():
    return {
        "Logistic regression (linear)": (*_logit(False), "roc_auc"),
        "Logistic regression (quadratic)": (*_logit(True), "roc_auc"),
        "XGBoost": (XGBClassifier(**XGB_FIXED, eval_metric="logloss"), XGB_GRID, "roc_auc"),
    }


def regressors():
    return {
        "Linear (ridge)": (*_ridge(False), "neg_mean_absolute_error"),
        "Quadratic (ridge)": (*_ridge(True), "neg_mean_absolute_error"),
        "XGBoost": (XGBRegressor(**XGB_FIXED), XGB_GRID, "neg_mean_absolute_error"),
    }


# ----------------------------------------------------------------------------- metrics
def cls_metrics(y, score, thresh=0.5, higher_is_comfort=True) -> dict:
    s = score if higher_is_comfort else -score
    pred = (score >= thresh).astype(int) if higher_is_comfort else (score <= thresh).astype(int)
    return {"accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"),
            "roc_auc": roc_auc_score(y, s)}


def matched_pred(score: np.ndarray, y: np.ndarray, splits) -> np.ndarray:
    """Top-share rule: within each test fold, call the share of rows equal to the TRAINING comfortable
    prevalence 'comfortable' (highest scores). Uses no test labels and treats every model, PMV included,
    the same way, so macro F1 is not dominated by one model predicting 'comfortable' for everything."""
    pred = np.zeros(len(score), int)
    for tr, te in splits:
        prev = y[tr].mean()
        thr = np.quantile(score[te], 1 - prev)
        pred[te] = (score[te] >= thr).astype(int)
    return pred


def sens_metrics(vote, vote_rounded, pred) -> dict:
    pred = np.clip(pred, -3, 3)
    return {"mae": mean_absolute_error(vote, pred),
            "within_1": float((np.abs(np.round(pred) - vote_rounded) <= 1).mean())}


def cluster_bootstrap(groups, fn, n_boot=config.N_BOOT, seed=config.SEED) -> dict:
    """95% CI by resampling whole buildings; fn(idx) -> dict of metrics."""
    rng = np.random.default_rng(seed)
    ids = pd.unique(groups)
    where = {g: np.flatnonzero(groups == g) for g in ids}
    res = []
    for _ in range(n_boot):
        pick = rng.choice(ids, size=len(ids), replace=True)
        idx = np.concatenate([where[g] for g in pick])
        try:
            res.append(fn(idx))
        except ValueError:  # resample with a single class
            continue
    r = pd.DataFrame(res)
    return {k: (r[k].quantile(0.025), r[k].quantile(0.975)) for k in r.columns}


# ----------------------------------------------------------------------------- cross-validation
def _fit_predict(name, est, grid, scoring, Xtr, ytr, gtr, Xte, kind):
    inner = GroupKFold(n_splits=3)
    gs = GridSearchCV(est, grid, scoring=scoring, cv=inner, n_jobs=1, refit=True)
    gs.fit(Xtr, ytr, groups=gtr)
    if kind == "cls":
        return gs.predict_proba(Xte)[:, 1], gs.best_params_
    return gs.predict(Xte), gs.best_params_


def oof_predictions(df: pd.DataFrame, splits, kind: str):
    """Pooled out-of-fold predictions for every model of the given kind ('cls' or 'reg')."""
    X = df[FEATURES].to_numpy(float)
    y = df[Y].to_numpy() if kind == "cls" else df[SENS].to_numpy(float)
    g = df[GROUP].to_numpy()
    models = classifiers() if kind == "cls" else regressors()
    oof = {n: np.full(len(df), np.nan) for n in models}
    folds = {n: np.full(len(df), -1) for n in models}
    best = {n: [] for n in models}
    for k, (tr, te) in enumerate(splits):
        for n, (est, grid, scoring) in models.items():
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                p, bp = _fit_predict(n, est, grid, scoring, X[tr], y[tr], g[tr], X[te], kind)
            oof[n][te] = p
            folds[n][te] = k
            best[n].append({kk: (vv.item() if hasattr(vv, "item") else vv) for kk, vv in bp.items()})
    return oof, folds, best


def building_splits(df: pd.DataFrame):
    sgk = StratifiedGroupKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.SEED)
    return list(sgk.split(df[FEATURES], df[Y], df[GROUP]))


def contributor_splits(df: pd.DataFrame):
    return list(LeaveOneGroupOut().split(df[FEATURES], groups=df[M["contributor"]]))


# ----------------------------------------------------------------------------- main experiment
def _ci_cols(row: dict, ci: dict) -> None:
    for k, (lo, hi) in ci.items():
        row[f"{k}_ci95"] = f"[{lo:.3f}, {hi:.3f}]"


def run_experiment(df: pd.DataFrame, splits, label: str, signed_pmv: bool = False):
    """Return (comfort_table, sensation_table, fold_table, oof_frame, best_params)."""
    pmv = pmv_baseline(df)
    y = df[Y].to_numpy()
    vote = df[SENS].to_numpy(float)
    vr = df["vote_rounded"].to_numpy()
    g = df[GROUP].to_numpy()
    n = len(df)

    oof_c, fold_c, best_c = oof_predictions(df, splits, "cls")
    oof_r, fold_r, best_r = oof_predictions(df, splits, "reg")

    # ---- comfort (binary) -------------------------------------------------
    cls_rows, fold_rows = [], []

    def add_cls(model, fn, scores, thresh, fold_ids, matched=True):
        mp = matched_pred(scores, y, splits) if matched else None
        full = (lambda i: {**fn(i), "macro_f1_matched": f1_score(y[i], mp[i], average="macro", zero_division=0)}) \
            if matched else (lambda i: {**fn(i), "macro_f1_matched": np.nan})
        row = {"validation": label, "model": model, **full(np.arange(n))}
        _ci_cols(row, cluster_bootstrap(g, full))
        cls_rows.append(row)
        if fold_ids is not None:
            for f in sorted(set(fold_ids[fold_ids >= 0])):
                m = fold_ids == f
                if len(np.unique(y[m])) == 2:
                    fold_rows.append({"validation": label, "model": model, "task": "comfort", "fold": int(f),
                                      "n": int(m.sum()), **cls_metrics(y[m], scores[m], thresh)})

    ref_folds = np.full(n, -1)
    for k, (_, te) in enumerate(splits):
        ref_folds[te] = k
    for name, col in (("PMV (ISO 7730)", "pmv_iso"), ("PMV (ASHRAE 55, cooling effect)", "pmv_ashrae")):
        if signed_pmv:      # positive class = warm-side discomfort: predicted when PMV >= +0.5
            a, th = pmv[col].to_numpy(), 0.5
        else:               # positive class = comfortable: predicted when |PMV| <= 1
            a, th = -np.abs(pmv[col].to_numpy()), -1.0
        add_cls(name, (lambda a, th: lambda i: cls_metrics(y[i], a[i], th))(a, th), a, th, ref_folds)
    for name, p in oof_c.items():
        add_cls(name, (lambda p: lambda i: cls_metrics(y[i], p[i]))(p), p, 0.5, fold_c[name])
    add_cls("Always predict the majority class (reference)",
            lambda i: {"accuracy": max(y[i].mean(), 1 - y[i].mean()),
                       "macro_f1": f1_score(y[i], np.full(len(i), int(y.mean() >= 0.5)), average="macro", zero_division=0),
                       "roc_auc": 0.5},
            np.ones(n), 0.5, None, matched=False)

    # ---- sensation vote (regression) -------------------------------------
    sens_rows = []

    def add_sens(model, pred, fold_ids):
        fn = lambda i: sens_metrics(vote[i], vr[i], pred[i])
        row = {"validation": label, "model": model, **fn(np.arange(n))}
        _ci_cols(row, cluster_bootstrap(g, fn))
        sens_rows.append(row)
        if fold_ids is not None:
            for f in sorted(set(fold_ids[fold_ids >= 0])):
                m = fold_ids == f
                fold_rows.append({"validation": label, "model": model, "task": "sensation", "fold": int(f),
                                  "n": int(m.sum()), **sens_metrics(vote[m], vr[m], pred[m])})

    add_sens("PMV (ISO 7730)", pmv["pmv_iso"].to_numpy(), None)
    add_sens("PMV (ASHRAE 55, cooling effect)", pmv["pmv_ashrae"].to_numpy(), None)
    for name, p in oof_r.items():
        add_sens(name, p, fold_r[name])
    add_sens("Predict training mean (reference)", np.full(n, vote.mean()), None)

    oof = df[[M["record_id"], GROUP, Y, "vote_rounded", SENS]].copy()
    oof = pd.concat([oof, pmv], axis=1)
    for name, p in oof_c.items():
        oof["p_comfort__" + name] = p
    for name, p in oof_r.items():
        oof["sens_pred__" + name] = p
    return (pd.DataFrame(cls_rows), pd.DataFrame(sens_rows), pd.DataFrame(fold_rows), oof,
            {"classifiers": best_c, "regressors": best_r})


def save_params(best: dict) -> None:
    (config.RESULTS / "best_params.json").write_text(json.dumps(best, indent=2, default=str))

In [ ]:
%%writefile src/explain.py
"""Final models, SHAP explanations and held-out permutation importance."""
from __future__ import annotations

import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold

try:
    from . import config, explore, models
except ImportError:  # pragma: no cover
    import config, explore, models  # type: ignore

LABELS = {"clo": "Clothing insulation (clo)", "ta": "Air temperature (degC)", "rh": "Relative humidity (%)",
          "vel": "Air speed (m/s)", "met": "Metabolic rate (met)"}
NICE = {"clo": "Clothing insulation", "ta": "Air temperature", "rh": "Relative humidity",
        "vel": "Air speed", "met": "Metabolic rate"}


def fit_final(df: pd.DataFrame):
    """Tune on all data with building-grouped 5-fold CV (final fit only; the honest estimates come from the
    nested validation in stage 3/4). Returns ({name: fitted estimator}, {name: best params})."""
    X, y, g = df[models.FEATURES].to_numpy(float), df[models.Y].to_numpy(), df[models.GROUP].to_numpy()
    fitted, params = {}, {}
    for name, key in (("Logistic regression (linear)", "Logistic regression (linear)"), ("XGBoost", "XGBoost")):
        est, grid, scoring = models.classifiers()[key]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            gs = GridSearchCV(est, grid, scoring=scoring, cv=GroupKFold(5), refit=True).fit(X, y, groups=g)
        fitted[name], params[name] = gs.best_estimator_, gs.best_params_
    return fitted, params


def shap_xgb(model, X: pd.DataFrame) -> np.ndarray:
    return shap.TreeExplainer(model).shap_values(X)


def shap_linear(pipe, X: pd.DataFrame) -> np.ndarray:
    scaler, lr = pipe.steps[0][1], pipe.steps[-1][1]
    Xs = scaler.transform(X)
    return shap.LinearExplainer(lr, shap.maskers.Independent(Xs, max_samples=len(Xs))).shap_values(Xs)


def global_importance(sv_x: np.ndarray, sv_l: np.ndarray, cols) -> pd.DataFrame:
    t = pd.DataFrame({"feature": cols, "mean_abs_shap_xgboost": np.abs(sv_x).mean(0),
                      "mean_abs_shap_logistic": np.abs(sv_l).mean(0)})
    for c in ("xgboost", "logistic"):
        t[f"share_{c}_pct"] = 100 * t[f"mean_abs_shap_{c}"] / t[f"mean_abs_shap_{c}"].sum()
    return t.sort_values("mean_abs_shap_xgboost", ascending=False).reset_index(drop=True)


def permutation_importance(df: pd.DataFrame, params: dict, splits, n_rep: int = 30) -> pd.DataFrame:
    """Held-out ROC AUC drop when one feature is shuffled in the test fold (leave-one-contributor-out)."""
    X, y = df[models.FEATURES].to_numpy(float), df[models.Y].to_numpy()
    rng = np.random.default_rng(config.SEED)
    rows = []
    for name in ("Logistic regression (linear)", "XGBoost"):
        est = models.classifiers()[name][0]
        est = clone(est).set_params(**params[name])
        for k, (tr, te) in enumerate(splits):
            if len(np.unique(y[te])) < 2:
                continue
            est.fit(X[tr], y[tr])
            base = roc_auc_score(y[te], est.predict_proba(X[te])[:, 1])
            for j, f in enumerate(models.FEATURES):
                drops = []
                for _ in range(n_rep):
                    Xp = X[te].copy()
                    Xp[:, j] = rng.permutation(Xp[:, j])
                    drops.append(base - roc_auc_score(y[te], est.predict_proba(Xp)[:, 1]))
                rows.append({"model": name, "fold": k, "feature": f, "auc_drop": float(np.mean(drops))})
    r = pd.DataFrame(rows)
    return r.groupby(["model", "feature"])["auc_drop"].agg(["mean", "std"]).reset_index()


# ------------------------------------------------------------------ figures
def fig_shap_global(imp: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    t = imp.sort_values("mean_abs_shap_xgboost")
    y = np.arange(len(t))
    ax.barh(y + 0.2, t["mean_abs_shap_xgboost"], 0.38, color=explore.C["alt"], label="XGBoost")
    ax.barh(y - 0.2, t["mean_abs_shap_logistic"], 0.38, color=explore.C["main"], label="Logistic regression")
    ax.set_yticks(y, [NICE[f] for f in t["feature"]])
    ax.set_xlabel("Mean |SHAP value| (log-odds of warm-side discomfort)")
    ax.set_title("Global feature importance")
    ax.legend(frameon=False, loc="lower right")
    explore.save(fig, "fig09_shap_global_importance")


def fig_shap_beeswarm(sv: np.ndarray, X: pd.DataFrame):
    plt.figure(figsize=(6.2, 3.8))
    shap.summary_plot(sv, X.rename(columns=NICE), show=False, plot_size=None)
    ax = plt.gca()
    ax.set_xlabel("SHAP value (log-odds of warm-side discomfort)")
    ax.set_title("XGBoost: how each feature moves the prediction")
    fig = plt.gcf()
    explore.save(fig, "fig10_shap_beeswarm_xgboost")


def fig_shap_dependence(sv: np.ndarray, X: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.6))
    for ax, f in zip(axes.ravel(), ["clo", "ta", "rh", "vel"]):
        j = list(X.columns).index(f)
        ax.scatter(X[f], sv[:, j], s=6, alpha=0.25, color=explore.C["grey"])
        bins = pd.qcut(X[f], 12, duplicates="drop")
        m = pd.DataFrame({"x": X[f], "s": sv[:, j], "b": bins}).groupby("b", observed=True).mean()
        ax.plot(m["x"], m["s"], color=explore.C["alt"], lw=2)
        ax.axhline(0, color="black", lw=0.7)
        ax.set_xlabel(LABELS[f])
        ax.set_ylabel("SHAP value (log-odds)")
    fig.suptitle("XGBoost SHAP dependence (line = binned mean)", y=1.0)
    fig.tight_layout()
    explore.save(fig, "fig11_shap_dependence")

In [ ]:
%%writefile src/scenarios.py
"""What-if scenarios: predicted warm-side discomfort across total clo, temperature, RH and air speed.

Insulation axis: ASHRAE `clo` is clothing only, while the manikin totals include bedding. Scenarios therefore use
TOTAL clo = garment clo + manikin cover increment (cover set minus 'Without cover', same posture). Garment clo is
a grid (quartiles of the data), not named garments: no garment clo values exist in the datasets.

Rules: ML "stays comfortable" = predicted P(warm-side discomfort) <= config.WARM_P_MAX (0.20);
PMV "stays comfortable" = PMV < config.PMV_WARM_CUT (+0.5). Sleep met = config.SLEEP_MET.
"""
from __future__ import annotations

import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone

try:
    from . import config, explore, models
except ImportError:  # pragma: no cover
    import config, explore, models  # type: ignore

F = models.FEATURES  # [clo, ta, rh, vel, met]
CLO_GRID = np.round(np.arange(0.10, 2.501, 0.01), 2)


def frame(clo, ta, rh, vel, met=None) -> pd.DataFrame:
    met = config.SLEEP_MET if met is None else met
    n = max(len(np.atleast_1d(a)) for a in (clo, ta, rh, vel, met))
    return pd.DataFrame({F[0]: np.broadcast_to(clo, n), F[1]: np.broadcast_to(ta, n), F[2]: np.broadcast_to(rh, n),
                         F[3]: np.broadcast_to(vel, n), F[4]: np.broadcast_to(met, n)}).astype(float)


def p_warm(model, X: pd.DataFrame) -> np.ndarray:
    return model.predict_proba(X[F].to_numpy(float))[:, 1]


def pmv(X: pd.DataFrame) -> pd.DataFrame:
    from pythermalcomfort.models import pmv_ppd_ashrae, pmv_ppd_iso
    from pythermalcomfort.utilities import v_relative
    ta = X[F[1]].to_numpy(float)
    vr = np.asarray(v_relative(v=X[F[3]].to_numpy(float), met=X[F[4]].to_numpy(float)), float)
    kw = dict(tdb=ta, tr=ta, vr=vr, rh=X[F[2]].to_numpy(float), met=X[F[4]].to_numpy(float),
              clo=X[F[0]].to_numpy(float), limit_inputs=False)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        iso, ash = pmv_ppd_iso(**kw), pmv_ppd_ashrae(**kw)
    return pd.DataFrame({"pmv_iso": np.asarray(iso.pmv, float), "pmv_ashrae": np.asarray(ash.pmv, float)})


def fan_levels(df: pd.DataFrame) -> dict:
    """Air speeds for 'no fan' / 'fan': median measured speed in rows where the fan flag is 0 / 1."""
    f = df[config.COLUMN_MAP["ashrae"]["fan"]]
    v = df[config.COLUMN_MAP["ashrae"]["air_speed"]]
    return {"no fan": float(v[f == 0].median()), "fan": float(v[f == 1].median()),
            "n_no_fan": int((f == 0).sum()), "n_fan": int((f == 1).sum())}


def cover_increments(clo_tbl: pd.DataFrame) -> pd.DataFrame:
    t = clo_tbl.copy()
    base = t[t["set"] == "Without cover"].set_index("posture")["clo_total"]
    t["increment"] = t.apply(lambda r: r["clo_total"] - base[r["posture"]], axis=1)
    return t[["set", "posture", "clo_total", "increment"]]


def highest_ok(clo_grid, ok: np.ndarray) -> float:
    """Highest clo such that the rule holds at EVERY clo from the grid start up to it (nan if it fails at the start)."""
    bad = np.flatnonzero(~ok)
    if len(bad) == 0:
        return float(clo_grid[-1])
    if bad[0] == 0:
        return float("nan")
    return float(clo_grid[bad[0] - 1])


def curve_table(models_by_name: dict, ta, rh, vel, label) -> pd.DataFrame:
    X = frame(CLO_GRID, ta, rh, vel)
    out = pd.DataFrame({"clo_total": CLO_GRID, "condition": label})
    for n, m in models_by_name.items():
        out[f"p_warm__{n}"] = p_warm(m, X)
    out = pd.concat([out, pmv(X)], axis=1)
    return out


def limits(curves: pd.DataFrame, primary: str, support_max: float) -> dict:
    r = {}
    for col, rule in ((f"p_warm__{primary}", lambda c: c <= config.WARM_P_MAX),
                      ("p_warm__XGBoost", lambda c: c <= config.WARM_P_MAX),
                      ("pmv_iso", lambda c: c < config.PMV_WARM_CUT),
                      ("pmv_ashrae", lambda c: c < config.PMV_WARM_CUT)):
        if col in curves:
            h = highest_ok(curves["clo_total"].to_numpy(), rule(curves[col].to_numpy()))
            r[col] = {"highest_clo": h, "beyond_data_range": bool(h > support_max) if h == h else False}
    return r


def bootstrap_limit(df: pd.DataFrame, primary_est, scen: dict, n_boot=config.N_BOOT_SCEN) -> dict:
    """Refit the primary model on building-resampled data; return percentile bands of the P(warm) curves and of the
    highest-comfortable-clo limit for each scenario in `scen` = {label: (ta, rh, vel)}."""
    rng = np.random.default_rng(config.SEED)
    ids = df[models.GROUP].unique()
    where = {g: np.flatnonzero(df[models.GROUP].to_numpy() == g) for g in ids}
    X, y = df[F].to_numpy(float), df[models.Y].to_numpy()
    curves = {k: [] for k in scen}
    lim = {k: [] for k in scen}
    for _ in range(n_boot):
        pick = rng.choice(ids, size=len(ids), replace=True)
        idx = np.concatenate([where[g] for g in pick])
        m = clone(primary_est).fit(X[idx], y[idx])
        for k, (ta, rh, vel) in scen.items():
            p = m.predict_proba(frame(CLO_GRID, ta, rh, vel)[F].to_numpy(float))[:, 1]
            curves[k].append(p)
            lim[k].append(highest_ok(CLO_GRID, p <= config.WARM_P_MAX))
    res = {}
    for k in scen:
        c = np.array(curves[k]); l = np.array(lim[k], float)
        res[k] = {"lo": np.percentile(c, 2.5, 0), "hi": np.percentile(c, 97.5, 0),
                  "limit_median": float(np.nanmedian(l)) if np.isfinite(l).any() else float("nan"),
                  "limit_lo": float(np.nanpercentile(l, 2.5)) if np.isfinite(l).any() else float("nan"),
                  "limit_hi": float(np.nanpercentile(l, 97.5)) if np.isfinite(l).any() else float("nan"),
                  "share_none": float(np.mean(~np.isfinite(l))), "share_at_grid_top": float(np.mean(l >= CLO_GRID[-1]))}
    return res


def support(df: pd.DataFrame, ta, rh, vel) -> dict:
    M = config.COLUMN_MAP["ashrae"]
    near = (df[M["air_temp"]].sub(ta).abs() <= 1) & (df[M["rel_humidity"]].sub(rh).abs() <= 7)
    return {"rows_within_1C_and_7pctRH": int(near.sum()),
            "rows_within_1C_7pctRH_and_vel_within_0.2": int((near & df[M["air_speed"]].sub(vel).abs().le(0.2)).sum()),
            "rows_met_le_0.8": int((df[M["met"]] <= 0.8).sum())}


# ------------------------------------------------------------------ figures
def fig_curves(curves: dict, boot: dict, primary: str, support_p99: float, fan: dict):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
    cols = {"no fan": explore.C["main"], "fan": explore.C["alt"]}
    ax = axes[0]
    for lab, c in curves.items():
        col = cols[lab]
        ax.plot(c["clo_total"], 100 * c[f"p_warm__{primary}"], color=col, label=f"{lab} ({fan[lab]:.2f} m/s)")
        ax.fill_between(c["clo_total"], 100 * boot[lab]["lo"], 100 * boot[lab]["hi"], color=col, alpha=0.15)
    ax.axhline(100 * config.WARM_P_MAX, ls="--", color="black", lw=0.8)
    ax.axvspan(support_p99, CLO_GRID[-1], color="grey", alpha=0.12)
    ax.set_xlim(0.1, 2.2); ax.set_ylim(0, 100)
    ax.set_xlabel("Total insulation, garment + bedding (clo)")
    ax.set_ylabel("Predicted warm-side discomfort (%)")
    ax.set_title("Data-driven model (95% band over buildings)")
    ax.text(support_p99 + 0.03, 92, "beyond 99th pct\nof training clo", fontsize=7, va="top")
    ax.legend(frameon=False, loc="upper left")
    ax = axes[1]
    for lab, c in curves.items():
        ax.plot(c["clo_total"], c["pmv_iso"], color=cols[lab], label=f"{lab} ISO 7730")
        ax.plot(c["clo_total"], c["pmv_ashrae"], color=cols[lab], ls=":", label=f"{lab} ASHRAE 55")
    ax.axhline(config.PMV_WARM_CUT, ls="--", color="black", lw=0.8)
    ax.set_xlim(0.1, 2.2)
    ax.set_xlabel("Total insulation, garment + bedding (clo)")
    ax.set_ylabel("PMV (+3 hot)")
    ax.set_title("PMV theory (not fitted to these votes)")
    ax.legend(frameon=False, fontsize=7, loc="upper left")
    fig.suptitle("29 degC, 85% RH, sleeping met", y=1.02, fontsize=11)
    fig.tight_layout()
    explore.save(fig, "fig12_scenario_curves_29C_85RH")


def fig_heatmaps(model, fan: dict, clo_ref: float):
    ta = np.arange(25, 34.01, 0.5); rh = np.arange(50, 90.01, 2.5)
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9), sharey=True)
    for ax, lab in zip(axes, ("no fan", "fan")):
        T, R = np.meshgrid(ta, rh)
        p = p_warm(model, frame(clo_ref, T.ravel(), R.ravel(), fan[lab])).reshape(T.shape)
        im = ax.pcolormesh(T, R, 100 * p, cmap="RdBu_r", vmin=0, vmax=60, shading="auto")
        ax.contour(T, R, p, levels=[config.WARM_P_MAX], colors="black", linewidths=1.2)
        ax.plot(29, 85, "k*", ms=10)
        ax.set_xlabel("Air temperature (degC)")
        ax.set_title(f"{lab} ({fan[lab]:.2f} m/s)")
    axes[0].set_ylabel("Relative humidity (%)")
    fig.colorbar(im, ax=axes, label="Predicted warm-side discomfort (%)")
    fig.suptitle(f"Garment clo {clo_ref:.2f}; black line = 20% rule; star = 29 degC / 85% RH", y=1.02, fontsize=11)
    explore.save(fig, "fig13_scenario_heatmaps_temp_rh")


def fig_sets(tab: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9), sharey=True)
    sets = list(tab["set"].unique())
    for ax, lab in zip(axes, ("no fan", "fan")):
        w = 0.38
        for k, (post, col) in enumerate((("supine", explore.C["main"]), ("side", explore.C["alt"]))):
            t = tab[(tab["condition"] == lab) & (tab["posture"] == post)].set_index("set").loc[sets]
            ax.bar(np.arange(len(sets)) + (k - 0.5) * w, 100 * t["p_warm_primary"], w, color=col,
                   label="Back (supine)" if post == "supine" else "Left side")
        ax.axhline(100 * config.WARM_P_MAX, ls="--", color="black", lw=0.8)
        ax.set_xticks(range(len(sets)), sets, rotation=15)
        ax.set_title(lab)
    axes[0].set_ylabel("Predicted warm-side discomfort (%)")
    axes[0].legend(frameon=False)
    fig.suptitle("29 degC, 85% RH, typical garment clo plus manikin cover increment", y=1.02, fontsize=11)
    fig.tight_layout()
    explore.save(fig, "fig14_scenario_bedding_sets")

In [ ]:
%%writefile src/stage4.py
"""Stage 4 driver: warm-side models vs PMV, SHAP, permutation importance, scenarios. Writes results/ and figures/."""
from __future__ import annotations

import json
import warnings

import numpy as np
import pandas as pd

try:
    from . import config, explain, explore, models, scenarios as sc
except ImportError:  # pragma: no cover
    import config, explain, explore, models, scenarios as sc  # type: ignore

warnings.filterwarnings("ignore")


def run(verbose: bool = True) -> dict:
    R = config.RESULTS
    R.mkdir(exist_ok=True)
    out: dict = {}
    df = models.set_target(models.load_clean(), config.TARGET_KIND)
    M = config.COLUMN_MAP["ashrae"]
    out["n_rows"], out["n_buildings"] = int(len(df)), int(df[models.GROUP].nunique())
    out["warm_prevalence"] = float(df[models.Y].mean())

    # ---- 1. model comparison (warm-side discomfort) --------------------------------------------
    cb, sb, fb, ob, bb = models.run_experiment(df, models.building_splits(df), "building 5-fold", signed_pmv=True)
    cl, sl, fl, ol, bl = models.run_experiment(df, models.contributor_splits(df), "leave-one-contributor-out",
                                               signed_pmv=True)
    cls = pd.concat([cb, cl], ignore_index=True); sens = pd.concat([sb, sl], ignore_index=True)
    cls.to_csv(R / "model_comparison_warm.csv", index=False)
    sens.to_csv(R / "model_comparison_sensation.csv", index=False)
    pd.concat([fb, fl]).to_csv(R / "fold_metrics.csv", index=False)
    ob.to_csv(R / "oof_predictions_building_cv.csv.gz", index=False)
    models.save_params({"building_5fold": bb, "leave_one_contributor_out": bl})
    out["cls"], out["sens"] = cls, sens

    ml = ["Logistic regression (linear)", "XGBoost"]
    auc = cls[cls.model.isin(ml)].pivot(index="model", columns="validation", values="roc_auc")
    primary = auc.mean(axis=1).idxmax()
    out["primary_model"] = primary
    out["auc_by_validation"] = auc

    explore.fig_roc_warm(ob)
    explore.fig_calibration_warm(ob)

    # ---- 2. final fits, SHAP, permutation importance -----------------------------------------------
    fitted, params = explain.fit_final(df)
    out["final_params"] = params
    X = df[models.FEATURES]
    sv_x = explain.shap_xgb(fitted["XGBoost"], X)
    sv_l = explain.shap_linear(fitted["Logistic regression (linear)"], X)
    imp = explain.global_importance(sv_x, sv_l, list(X.columns))
    imp.to_csv(R / "shap_global_importance.csv", index=False)
    pi = explain.permutation_importance(df, params, models.contributor_splits(df))
    pi.to_csv(R / "permutation_importance_loco.csv", index=False)
    lr = fitted["Logistic regression (linear)"].steps[-1][1]
    coef = pd.DataFrame({"feature": models.FEATURES, "std_coef_log_odds": lr.coef_[0]})
    coef["odds_ratio_per_sd"] = np.exp(coef["std_coef_log_odds"])
    sd = df[models.FEATURES].std().to_numpy()
    coef["sd"] = sd
    coef.to_csv(R / "logistic_coefficients.csv", index=False)
    explain.fig_shap_global(imp); explain.fig_shap_beeswarm(sv_x, X); explain.fig_shap_dependence(sv_x, X)
    out["shap"], out["perm"], out["coef"] = imp, pi, coef

    # ---- 3. scenarios ----------------------------------------------------------------------------------
    clo_tbl = pd.read_csv(R / "manikin_total_clo.csv")
    incr = sc.cover_increments(clo_tbl); incr.to_csv(R / "manikin_cover_increments.csv", index=False)
    fan = sc.fan_levels(df); out["fan"] = fan
    p99 = float(df[M["clo"]].quantile(0.99)); p95 = float(df[M["clo"]].quantile(0.95))
    q = df[M["clo"]].quantile([0.25, 0.5, 0.75]).round(2).tolist()
    out["clo_quantiles"] = {"p25": q[0], "p50": q[1], "p75": q[2], "p95": p95, "p99": p99}
    est = {n: fitted[n] for n in ml}
    TA, RH = 29.0, 85.0
    out["support_29_85"] = {lab: sc.support(df, TA, RH, v) for lab, v in ((k, fan[k]) for k in ("no fan", "fan"))}

    curves = {lab: sc.curve_table(est, TA, RH, fan[lab], lab) for lab in ("no fan", "fan")}
    pd.concat(curves.values()).to_csv(R / "scenario_curves_29C_85RH.csv", index=False)
    lim = {lab: sc.limits(curves[lab], primary, p99) for lab in curves}
    out["limits_29_85"] = lim

    primary_est = est[primary]
    scen = {lab: (TA, RH, fan[lab]) for lab in curves}
    boot = sc.bootstrap_limit(df, primary_est, scen)
    out["boot_29_85"] = {k: {kk: vv for kk, vv in v.items() if kk not in ("lo", "hi")} for k, v in boot.items()}
    sc.fig_curves(curves, boot, primary, p99, fan)
    sc.fig_heatmaps(est[primary], fan, q[1])

    # limits across temperature (RH 85) and across RH (29 C)
    rows = []
    for ta in (27, 28, 29, 30, 31):
        for lab in ("no fan", "fan"):
            c = sc.curve_table(est, ta, RH, fan[lab], lab)
            l = sc.limits(c, primary, p99)
            rows.append({"ta": ta, "rh": RH, "fan": lab, **{f"{k}": v["highest_clo"] for k, v in l.items()},
                         "support_rows": sc.support(df, ta, RH, fan[lab])["rows_within_1C_and_7pctRH"]})
    for rh in (60, 70, 80):
        for lab in ("no fan", "fan"):
            c = sc.curve_table(est, TA, rh, fan[lab], lab)
            l = sc.limits(c, primary, p99)
            rows.append({"ta": TA, "rh": rh, "fan": lab, **{f"{k}": v["highest_clo"] for k, v in l.items()},
                         "support_rows": sc.support(df, TA, rh, fan[lab])["rows_within_1C_and_7pctRH"]})
    lt = pd.DataFrame(rows); lt.to_csv(R / "scenario_highest_clo_by_conditions.csv", index=False)
    out["limits_table"] = lt

    # manikin sets: garment quartile + cover increment, both postures, both fan states
    rows = []
    for g_lab, g in (("p25", q[0]), ("median", q[1]), ("p75", q[2])):
        for _, r in incr.iterrows():
            tot = g + r["increment"]
            for lab in ("no fan", "fan"):
                X1 = sc.frame(tot, TA, RH, fan[lab])
                pm = sc.pmv(X1)
                rows.append({"garment_level": g_lab, "garment_clo": g, "set": r["set"], "posture": r["posture"],
                             "cover_increment": round(r["increment"], 3), "total_clo": round(tot, 3),
                             "condition": lab, "p_warm_primary": float(sc.p_warm(est[primary], X1)[0]),
                             "p_warm_xgboost": float(sc.p_warm(est["XGBoost"], X1)[0]),
                             "pmv_iso": float(pm["pmv_iso"][0]), "pmv_ashrae": float(pm["pmv_ashrae"][0]),
                             "total_above_p99": bool(tot > p99)})
    st = pd.DataFrame(rows); st.to_csv(R / "scenario_bedding_sets_29C_85RH.csv", index=False)
    sc.fig_sets(st[st.garment_level == "median"])
    out["sets"] = st

    # ---- 4. numbers file ---------------------------------------------------------------------------------
    def tojson(o):
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, (pd.DataFrame,)):
            return o.to_dict("records")
        return str(o)
    (R / "key_numbers.json").write_text(json.dumps({k: v for k, v in out.items() if k not in ("cls", "sens", "shap", "perm", "coef", "sets", "limits_table", "auc_by_validation")}, indent=2, default=tojson))
    return out

In [ ]:
%%writefile src/report.py
"""Generate results/summary.md and results/design_garment_budget.csv.

Every number is read from files written by the pipeline (results/*.csv, results/key_numbers.json) or computed here
from data/processed/ashrae_clean.csv.gz. Nothing numeric is typed by hand. Qualitative wording was checked against
these numbers when the template was written; the checks are asserted at the top of build_all().
"""
from __future__ import annotations

import json

import pandas as pd

try:
    from . import config
except ImportError:  # pragma: no cover
    import config  # type: ignore

R = config.RESULTS


def f2(x):
    return f"{x:.2f}"


def f3(x):
    return f"{x:.3f}"


def pct(x, d=1):
    return f"{100 * x:.{d}f}%"


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)


def load():
    d = {}
    d["flog"] = pd.read_csv(R / "filter_log.csv")
    d["cls"] = pd.read_csv(R / "model_comparison_warm.csv")
    d["sens"] = pd.read_csv(R / "model_comparison_sensation.csv")
    d["neg"] = pd.read_csv(R / "neg_comfort_label_model_comparison.csv")
    d["negs"] = pd.read_csv(R / "neg_comfort_label_sensation_comparison.csv")
    d["shap"] = pd.read_csv(R / "shap_global_importance.csv").set_index("feature")
    d["perm"] = pd.read_csv(R / "permutation_importance_loco.csv")
    d["coef"] = pd.read_csv(R / "logistic_coefficients.csv").set_index("feature")
    d["lt"] = pd.read_csv(R / "scenario_highest_clo_by_conditions.csv")
    d["sets"] = pd.read_csv(R / "scenario_bedding_sets_29C_85RH.csv")
    d["inc"] = pd.read_csv(R / "manikin_cover_increments.csv")
    d["clo"] = pd.read_csv(R / "manikin_total_clo.csv")
    d["cov"] = pd.read_csv(R / "clo_coverage.csv")
    d["country"] = pd.read_csv(R / "subset_by_country.csv")
    d["stats"] = pd.read_csv(R / "subset_summary_stats.csv", index_col=0)
    d["key"] = json.loads((R / "key_numbers.json").read_text())
    d["data"] = pd.read_csv(config.PROCESSED / "ashrae_clean.csv.gz")
    return d


def row(df, validation, model):
    return df[(df.validation == validation) & (df.model == model)].iloc[0]


def build_all() -> None:
    d = load()
    K = d["key"]
    BV, LO = "building 5-fold", "leave-one-contributor-out"
    P = K["primary_model"]
    LR, XG = "Logistic regression (linear)", "XGBoost"
    PI, PA = "PMV (ISO 7730)", "PMV (ASHRAE 55, cooling effect)"
    fan = K["fan"]
    lim = K["limits_29_85"]
    boot = K["boot_29_85"]
    q = K["clo_quantiles"]
    key_lr = f"p_warm__{P}"

    # ---- sanity checks on the claims the templates make ------------------------------------------
    sh = d["shap"]
    assert sh.index[0] == "ta" and sh.index[1] == "vel", "template assumes temperature > air speed > rest"
    assert sh.loc["clo", "share_xgboost_pct"] < 5 and sh.loc["clo", "share_logistic_pct"] < 6
    assert lim["fan"][key_lr]["highest_clo"] > lim["no fan"][key_lr]["highest_clo"]
    assert lim["fan"]["pmv_iso"]["highest_clo"] > lim["no fan"]["pmv_iso"]["highest_clo"]
    assert d["coef"].loc["ta", "odds_ratio_per_sd"] > 1 and d["coef"].loc["vel", "odds_ratio_per_sd"] < 1

    # ---- derived numbers ---------------------------------------------------------------------------
    f = d["flog"]
    n_final, b_final = int(f.iloc[-1].rows_after), int(f.iloc[-1].buildings_after)
    data = d["data"]
    M = config.COLUMN_MAP["ashrae"]
    near = (data[M["air_temp"]].sub(29).abs() <= 1) & (data[M["rel_humidity"]].sub(85).abs() <= 7)
    obs = {}
    for lab, flag in (("no fan", 0), ("fan", 1)):
        m = near & (data[M["fan"]] == flag)
        obs[lab] = (int(m.sum()), float(data.loc[m, models_y(data)].mean()) if m.sum() else float("nan"))
    inc = d["inc"]
    base_inc = inc.set_index(["set", "posture"])["increment"]
    posture_diff = (d["clo"].pivot(index="set", columns="posture", values="clo_total")
                    .assign(diff=lambda t: t["supine"] - t["side"])["diff"])
    sets = d["sets"]
    med = sets[sets.garment_level == "median"].copy()
    med["ML ok"] = med["p_warm_primary"] <= config.WARM_P_MAX
    med["PMV ISO ok"] = med["pmv_iso"] < config.PMV_WARM_CUT
    med["PMV ASHRAE ok"] = med["pmv_ashrae"] < config.PMV_WARM_CUT
    # garment budget = highest total clo (PMV ISO, the more cautious theory model) minus cover increment
    bud = []
    for lab in ("no fan", "fan"):
        for _, r in inc.iterrows():
            g = lim[lab]["pmv_iso"]["highest_clo"] - r["increment"]
            bud.append({"condition": lab, "set": r["set"], "posture": r["posture"],
                        "cover_increment": f2(r["increment"]),
                        "garment_clo_budget_pmv_iso": f2(max(g, 0.0)),
                        "fits_training_garment_range": "yes" if g >= 0.1 else "no garment fits"})
    bud = pd.DataFrame(bud)
    bud.to_csv(R / "design_garment_budget.csv", index=False)

    # ---- tables for the summary ----------------------------------------------------------------------
    def cmp_table(validation):
        t = d["cls"][d["cls"].validation == validation]
        out = pd.DataFrame({"Model": t.model, "Accuracy": t.accuracy.map(f3), "Macro F1": t.macro_f1.map(f3),
                            "Macro F1 (matched share)": t.macro_f1_matched.map(lambda v: "n/a" if v != v else f3(v)),
                            "ROC AUC": t.roc_auc.map(f3), "ROC AUC 95% CI": t.roc_auc_ci95})
        return md_table(out)

    def sens_table(validation):
        t = d["sens"][d["sens"].validation == validation]
        out = pd.DataFrame({"Model": t.model, "MAE (vote units)": t.mae.map(f3), "MAE 95% CI": t.mae_ci95,
                            "Accuracy within +-1": t.within_1.map(f3), "Within +-1 95% CI": t.within_1_ci95})
        return md_table(out)

    REF = "Always predict the majority class (reference)"   # reference row of the negative-result table
    if REF not in set(d["neg"].model):   # older stage 3 runs named the reference row differently
        REF = [m for m in d["neg"].model.unique() if m.startswith("Always")][0]
    neg_b = lambda m: row(d["neg"], BV, m)
    neg_l = lambda m: row(d["neg"], LO, m)

    imp = sh.reset_index()[["feature", "share_xgboost_pct", "share_logistic_pct"]].copy()
    imp["share_xgboost_pct"] = imp["share_xgboost_pct"].map(lambda v: f"{v:.1f}%")
    imp["share_logistic_pct"] = imp["share_logistic_pct"].map(lambda v: f"{v:.1f}%")
    imp.columns = ["Feature", "SHAP share, XGBoost", "SHAP share, logistic"]
    pm = d["perm"].pivot(index="feature", columns="model", values="mean").loc[sh.index]
    perm = pd.DataFrame({"Feature": pm.index, "AUC drop, logistic": pm[LR].map(f3).values,
                         "AUC drop, XGBoost": pm[XG].map(f3).values})

    lt = d["lt"].copy()
    def cell(v):
        return "none" if v != v else f2(v)
    lt_tab = pd.DataFrame({"Air temp (degC)": lt.ta.map(lambda v: f"{v:.0f}"), "RH (%)": lt.rh.map(lambda v: f"{v:.0f}"),
                           "Air speed": lt.fan, "Rows nearby": lt.support_rows,
                           "ML limit (clo)": lt[key_lr].map(cell), "PMV ISO limit": lt.pmv_iso.map(cell),
                           "PMV ASHRAE limit": lt.pmv_ashrae.map(cell)})
    st = med[["set", "posture", "total_clo", "condition", "p_warm_primary", "pmv_iso", "pmv_ashrae", "ML ok",
              "PMV ISO ok", "PMV ASHRAE ok"]].copy()
    st["p_warm_primary"] = st.p_warm_primary.map(lambda v: pct(v))
    for c in ("total_clo", "pmv_iso", "pmv_ashrae"):
        st[c] = st[c].map(f2)
    for c in ("ML ok", "PMV ISO ok", "PMV ASHRAE ok"):
        st[c] = st[c].map({True: "yes", False: "no"})
    st.columns = ["Set", "Posture", "Total clo", "Air", "ML P(warm)", "PMV ISO", "PMV ASHRAE", "ML ok", "ISO ok", "ASHRAE ok"]

    l29 = d["lt"][(d["lt"].ta == 29) & (d["lt"].fan == "no fan")].set_index("rh")
    hum_note = (f"Caution on humidity: at 29 degC without a fan the data-driven limit {cell(l29.loc[70, key_lr])} clo at 70% RH becomes "
                f"{cell(l29.loc[85, key_lr])} clo at 85% RH (it rises with humidity), while the PMV limit falls from {cell(l29.loc[70, 'pmv_iso'])} to "
                f"{cell(l29.loc[85, 'pmv_iso'])} clo. The fitted humidity effect is negative (odds ratio {f2(d['coef'].loc['rh','odds_ratio_per_sd'])} per sd), which "
                f"reflects who lives where in this subset, so do not read the data-driven humidity trend as physiology.")
    cov_rows = d["cov"]
    cov_min, cov_max = int(cov_rows.ashrae_rows_with_clo_ge_this.min()), int(cov_rows.ashrae_rows_with_clo_ge_this.max())
    n_fan_rows = fan["n_fan"] + fan["n_no_fan"]
    ctry = d["country"]
    ctry_txt = ", ".join(f"{r.country.title()} {int(r.rows):,} rows ({int(r.buildings)} buildings)"
                         for r in ctry.itertuples())

    # =================================================================================================
    summary = f"""# Results summary

All numbers below were produced by the code in `src/` (stage 4 driver: `src/stage4.py`) and are stored in `results/`.
Datasets (both CC BY 4.0): ASHRAE Global Thermal Comfort Database II v2.1 (https://zenodo.org/records/4993952) and
Heat balance of sleeping sets, Newton thermal manikin, Brno University of Technology (https://zenodo.org/records/20179324).

## 1. Data used (`results/filter_log.csv`)

{md_table(f[['step','rows_before','rows_after','rows_dropped','buildings_after']].rename(columns={'step':'Step','rows_before':'Rows before','rows_after':'Rows after','rows_dropped':'Dropped','buildings_after':'Buildings after'}))}

Final subset: {n_final:,} votes from {b_final} buildings, {data[M['contributor']].nunique()} contributors. {ctry_txt}.
There are no Sri Lankan records. Share of votes that are warm-side discomfort (rounded sensation vote of +1 or more):
{pct(K['warm_prevalence'])}.

## 2. Target definition (changed after the first modelling pass)

The first target, "comfortable = vote from -1 to +1", is almost unpredictable from the inputs (section 3, negative result).
It lumps warm and cold votes together. The main target is therefore **warm-side discomfort: rounded sensation vote of
+1 or more**, which is also the relevant failure for hot nights.

## 3. Negative result with the original "comfortable" label (`results/neg_comfort_label_*.csv`)

Held-out buildings: ROC AUC {f3(neg_b(XG).roc_auc)} (XGBoost), {f3(neg_b('Logistic regression (quadratic)').roc_auc)} (quadratic logistic),
{f3(neg_b(LR).roc_auc)} (linear logistic), {f3(neg_b(PA).roc_auc)} (PMV ASHRAE 55). Accuracy of every ML model
({f3(neg_b(XG).accuracy)} for XGBoost) equals always predicting "comfortable" ({f3(neg_b(REF).accuracy)}).
Leaving one contributor out, AUC fell to {f3(neg_l(LR).roc_auc)} (linear logistic), {f3(neg_l('Logistic regression (quadratic)').roc_auc)} (quadratic) and
{f3(neg_l(XG).roc_auc)} (XGBoost). Sensation MAE in that check was {f3(row(d['negs'], LO, 'Linear (ridge)').mae)} (ridge) to {f3(row(d['negs'], LO, 'XGBoost').mae)} (XGBoost) against {f3(row(d['negs'], LO, 'Predict training mean (reference)').mae)} for predicting the mean vote, with overlapping intervals, so there was essentially no gain.

## 4. Model comparison against PMV, main target (warm-side discomfort)

Inputs for all models: clo, air temperature, relative humidity, air speed, met (PMV also uses mean radiant temperature,
falling back to air temperature where missing). "Macro F1 (matched share)" calls the training-prevalence share of rows
in each test fold "warm" for every model, PMV included, using no test labels.

**Validation 1: building-grouped 5-fold** (StratifiedGroupKFold by `building_id`; intervals from resampling buildings)

{cmp_table(BV)}

**Validation 2: leave one contributor out** (4 held-out studies)

{cmp_table(LO)}

**Sensation vote (continuous), building-grouped 5-fold**

{sens_table(BV)}

**Sensation vote, leave one contributor out**

{sens_table(LO)}

Reading: the linear logistic model has the best or near-best ROC AUC in both validations ({f3(row(d['cls'], BV, LR).roc_auc)} and
{f3(row(d['cls'], LO, LR).roc_auc)}) and is the primary model. PMV ranks warm-side discomfort about as well (AUC {f3(row(d['cls'], BV, PI).roc_auc)}
for ISO 7730, {f3(row(d['cls'], BV, PA).roc_auc)} for ASHRAE 55) with overlapping intervals, but its raw threshold is badly
calibrated (accuracy {f3(row(d['cls'], BV, PI).accuracy)} at PMV >= +0.5, versus {f3(row(d['cls'], BV, 'Always predict the majority class (reference)').accuracy)}
for always predicting "not warm"). For the sensation vote, PMV has larger error (MAE {f3(row(d['sens'], BV, PA).mae)} to
{f3(row(d['sens'], BV, PI).mae)}) than predicting the mean vote ({f3(row(d['sens'], BV, 'Predict training mean (reference)').mae)}).
The ML models reduce MAE only slightly ({f3(row(d['sens'], BV, 'Linear (ridge)').mae)} for ridge).

## 5. What matters most (`results/shap_global_importance.csv`, `permutation_importance_loco.csv`, `logistic_coefficients.csv`)

Mean absolute SHAP value as a share of the total (final models fit on all {n_final:,} rows; SHAP describes the model, not causes):

{md_table(imp)}

Held-out permutation importance (drop in ROC AUC when one input is shuffled, leave-one-contributor-out folds):

{md_table(perm)}

Logistic odds ratio of warm-side discomfort per one standard deviation: air temperature {f2(d['coef'].loc['ta','odds_ratio_per_sd'])}
(sd {f2(d['coef'].loc['ta','sd'])} degC), air speed {f2(d['coef'].loc['vel','odds_ratio_per_sd'])} (sd {f2(d['coef'].loc['vel','sd'])} m/s),
relative humidity {f2(d['coef'].loc['rh','odds_ratio_per_sd'])} (sd {f2(d['coef'].loc['rh','sd'])} %), clo {f2(d['coef'].loc['clo','odds_ratio_per_sd'])}
(sd {f2(d['coef'].loc['clo','sd'])} clo). The dependence plots (`figures/fig11_shap_dependence`) show a steep rise with
temperature above roughly 27 degC, a fall with air speed up to about 1 m/s, a small negative association with humidity
(the opposite of what physiology alone suggests, so it reflects who lives where, not humidity as a cause), and a small,
irregular clo pattern. Clo contributes {sh.loc['clo','share_xgboost_pct']:.1f}% (XGBoost) and {sh.loc['clo','share_logistic_pct']:.1f}% (logistic) of total SHAP
importance, and shuffling it changes held-out AUC by about {d['perm'][(d['perm'].feature=='clo')]['mean'].abs().max():.3f} at most. **This data barely identifies an effect of clothing insulation.**

## 6. Sleepwear insulation from the manikin data (`results/manikin_total_clo.csv`, `manikin_cover_increments.csv`)

{md_table(d['clo'].assign(rtot_m2K_per_W=d['clo'].rtot_m2K_per_W.map(f3), clo_total=d['clo'].clo_total.map(f2), clo_uncertainty=d['clo'].clo_uncertainty.map(f2)).rename(columns={'set':'Set','posture':'Posture','rtot_m2K_per_W':'Rtot (m2K/W)','clo_total':'Total clo','clo_uncertainty':'Uncertainty (clo)'}))}

The "sets" are bedding configurations; the README does not say what garments were worn. Adding a cover to
"Without cover" adds {f2(inc[inc.increment>0].increment.min())} to {f2(inc.increment.max())} clo, while lying on the side instead of the back lowers the total by
{f2(posture_diff.min())} to {f2(posture_diff.max())} clo. Torso-segment back values above 5 clo (`manikin_torso_segments.csv`) look like a
contact artefact and were not used.

## 7. Scenarios at 29 degC and 85% RH (`results/scenario_*.csv`)

Scenario inputs: sleeping met {config.SLEEP_MET}; "no fan" = {f2(fan['no fan'])} m/s and "fan" = {f2(fan['fan'])} m/s (median measured speed in rows
where the fan flag is 0 / 1, from {fan['n_no_fan']} and {fan['n_fan']} rows). Total insulation = garment clo + manikin cover
increment. Rule for the data-driven model: predicted P(warm-side discomfort) <= {pct(config.WARM_P_MAX, 0)}. Rule for PMV: PMV < +{config.PMV_WARM_CUT}.
Support: {K['support_29_85']['no fan']['rows_within_1C_and_7pctRH']} votes lie within 1 degC and 7 % RH of this point
({obs['no fan'][0]} with the fan flag off, {obs['fan'][0]} with it on, the rest unrecorded), of which {pct(obs['no fan'][1], 0)} (no fan) and {pct(obs['fan'][1], 0)} (fan) were
warm-side discomfort. Only {K['support_29_85']['no fan']['rows_met_le_0.8']} votes have met of 0.8 or lower, so the sleeping-met input is near the edge of the data.

**Highest total clo that stays comfortable at 29 degC / 85% RH**

| Basis | No fan ({f2(fan['no fan'])} m/s) | Fan ({f2(fan['fan'])} m/s) |
|---|---|---|
| Data-driven model ({P}) | {f2(lim['no fan'][key_lr]['highest_clo'])} clo | {f2(lim['fan'][key_lr]['highest_clo'])} clo (beyond the data range: {lim['fan'][key_lr]['beyond_data_range']}) |
| ... 95% interval over buildings (bootstrap, {config.N_BOOT_SCEN} refits) | {f2(boot['no fan']['limit_lo'])} to {f2(boot['no fan']['limit_hi'])} clo | {f2(boot['fan']['limit_lo'])} to {f2(boot['fan']['limit_hi'])} clo |
| XGBoost (cross-check) | {f2(lim['no fan']['p_warm__XGBoost']['highest_clo'])} clo (top of grid, never exceeds {pct(config.WARM_P_MAX,0)}) | {f2(lim['fan']['p_warm__XGBoost']['highest_clo'])} clo (top of grid) |
| PMV, ISO 7730 | {f2(lim['no fan']['pmv_iso']['highest_clo'])} clo | {f2(lim['fan']['pmv_iso']['highest_clo'])} clo |
| PMV, ASHRAE 55 (cooling effect) | {f2(lim['no fan']['pmv_ashrae']['highest_clo'])} clo | {f2(lim['fan']['pmv_ashrae']['highest_clo'])} clo |

The training data's clo has a 95th percentile of {f2(q['p95'])} and a 99th percentile of {f2(q['p99'])}. The data-driven limits are
very wide (in {pct(boot['no fan']['share_at_grid_top'],0)} and {pct(boot['fan']['share_at_grid_top'],0)} of resamples the limit reaches the top of the {f2(2.5)} clo grid),
which is what a near-zero clo effect looks like. The PMV limits are physically derived, are not fitted to these votes, and
agree with the lower end of the data-driven interval. **Use the PMV limits as the working design numbers and the
data-driven result as support, not proof.**

**Highest comfortable total clo by condition** (`scenario_highest_clo_by_conditions.csv`; "none" = rule fails even at 0.10 clo;
rows nearby = votes within 1 degC and 7 % RH)

{md_table(lt_tab)}

{hum_note}

**Bedding sets at 29 degC / 85% RH, garment clo {f2(q['p50'])} (data median) plus manikin cover increment** (`scenario_bedding_sets_29C_85RH.csv`
also has the {f2(q['p25'])} and {f2(q['p75'])} garment levels)

{md_table(st)}

**Garment clo that is left once a cover is chosen** (highest total from PMV ISO 7730 minus the cover increment, `results/design_garment_budget.csv`)

{md_table(bud)}

## 8. Limitations

- ASHRAE votes come from awake people in buildings, not people sleeping, so they only stand in for sleep comfort.
- Neither dataset has skin-temperature data.
- The manikin data covers only the back (supine) and left side, and four bedding configurations with no garment detail.
- The data are observational: findings are associations, not causes (fans are used when it is hot; cooler homes have more clothing).
- The subset has {b_final} buildings but only {data[M['contributor']].nunique()} contributors, three countries and no Sri Lankan sites.
- ASHRAE `clo` is clothing only; manikin totals include bedding. Adding the cover increment to garment clo assumes the two are additive.
- {cov_min} to {cov_max} votes reach any manikin total clo, so totals above about {f2(q['p99'])} clo are extrapolation for the data-driven model.
- With the original label, ML models did not beat a constant predictor, and across-study performance collapsed (section 3).
- PMV was run outside its ISO 7730 validity range for many rows (limit checks switched off), and mean radiant temperature was missing for a large share of rows.
"""
    (R / "summary.md").write_text(summary)


def models_y(df: pd.DataFrame) -> str:
    """Name of the warm-side column in the processed file (derived from the rounded vote)."""
    if "warm" not in df:
        df["warm"] = (df["vote_rounded"] >= 1).astype(int)
    return "warm"

## 1. Download both datasets into `data/raw/`
File names are discovered from the Zenodo API, checksums are verified, zips are extracted.

In [ ]:
import importlib, config, download
importlib.reload(config); importlib.reload(download)
download.main()

## 2. Inspect: ASHRAE Global Thermal Comfort Database II

In [ ]:
import inspect_data
importlib.reload(inspect_data)
tables = {}
tables["ashrae"] = inspect_data.inspect_dataset("ashrae")

## 3. Inspect: manikin heat-balance data

In [ ]:
tables["manikin"] = inspect_data.inspect_dataset("manikin")

## 4. Candidate columns (hints only, to be confirmed by a human)

In [ ]:
cand = inspect_data.write_candidates(tables)
print(cand.to_string(index=False) if len(cand) else "No candidates found")

## 5. Package the inspection output
Raw data is **not** included (it is large and public); only the reports and Zenodo record metadata are zipped.

In [ ]:
import zipfile
out_zip = PROJECT / "stage1_outputs.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (PROJECT / "results" / "inspection").rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(PROJECT))
    for p in (PROJECT / "data" / "raw").glob("*/record_metadata.json"):
        z.write(p, p.relative_to(PROJECT))
    listing = "\n".join(f"{p.relative_to(PROJECT)}\t{p.stat().st_size}" for p in sorted((PROJECT / "data" / "raw").rglob("*")) if p.is_file())
    z.writestr("data_raw_file_listing.tsv", listing)
print("Wrote", out_zip, f"({out_zip.stat().st_size/1024:.0f} kB)")
if IN_COLAB:
    from google.colab import files

---
# Stage 2: filtering, cleaning, manikin clo, exploration
Decisions in force (set in `src/config.py`): hot-humid = climate in the explicit class list in `config.HOT_HUMID_CLASSES`; residential = `multifamily housing`; low activity = met <= 1.3; comfortable = rounded sensation vote in [-1, +1].

## 6. ASHRAE: climate classes, filters and cleaning log

In [ ]:
import pandas as pd
import clean
importlib.reload(config); importlib.reload(clean)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

raw = clean.load_ashrae()
print("\nClimate classes (all rows) and whether the hot-humid rule matches them:")
clim = clean.climate_report(raw)
print(clim.to_string(index=False))
print("\nBuilding types:"); print(raw[config.COLUMN_MAP["ashrae"]["building_type"]].value_counts(dropna=False).to_string())

In [ ]:
d, flog = clean.build_subset(raw)
print(flog.to_string(index=False))
sens = clean.met_sensitivity(raw)
print("\nSensitivity: rows after QA + hot-humid + residential, by met cutoff"); print(sens.to_string(index=False))

(config.RESULTS).mkdir(exist_ok=True)
flog.to_csv(config.RESULTS / "filter_log.csv", index=False)
clim.to_csv(config.RESULTS / "climate_classes.csv", index=False)
sens.to_csv(config.RESULTS / "met_sensitivity.csv", index=False)
print("\nSaved", clean.save_processed(d))

In [ ]:
M = config.COLUMN_MAP["ashrae"]
num = [M["air_temp"], M["rel_humidity"], M["air_speed"], M["clo"], M["met"], M["thermal_sensation"]]
summ = d[num].describe().T
print("Final subset:", f"{len(d):,} rows,", d[M["building_id"]].nunique(), "buildings,",
      d[M["contributor"]].nunique(), "contributors,", d[M["country"]].nunique(), "countries")
print(summ.to_string())
print("\nRounded vote counts:"); print(d["vote_rounded"].value_counts().sort_index().to_string())
print("\nShare comfortable (vote in [-1,+1]): %.3f" % d["comfortable"].mean())
print("\nRows per country / climate / building:")
print(d[M["country"]].value_counts().to_string()); print(d[M["climate"]].value_counts().to_string())
print(d.groupby(M["building_id"]).size().describe().to_string())
print("\nAcceptability available for", int(d[M["thermal_acceptability"]].notna().sum()), "rows;",
      "fan recorded for", int(d[M["fan"]].notna().sum()), "rows")
summ.to_csv(config.RESULTS / "subset_summary_stats.csv")
d.groupby(M["country"]).agg(rows=("comfortable", "size"), buildings=(M["building_id"], "nunique"),
                            share_comfortable=("comfortable", "mean")).to_csv(config.RESULTS / "subset_by_country.csv")

## 7. Manikin data: total clo by sleeping set and posture
The sets are bedding configurations, not garments. Torso-segment values above 5 clo are flagged as suspect.

In [ ]:
import manikin
importlib.reload(manikin)
clo_tbl, reps, seg = manikin.build_clo_table()
print(clo_tbl.to_string(index=False))
print("\nTorso segments (suspect_artifact = clo > 5):"); print(seg.to_string(index=False))
clo_tbl.to_csv(config.RESULTS / "manikin_total_clo.csv", index=False)
reps.to_csv(config.RESULTS / "manikin_replicates.csv", index=False)
seg.to_csv(config.RESULTS / "manikin_torso_segments.csv", index=False)

# How much of the ASHRAE training data reaches the manikin clo range?
rows = []
for _, r in clo_tbl.iterrows():
    rows.append({"set": r["set"], "posture": r["posture"], "clo_total": r["clo_total"],
                 "ashrae_rows_with_clo_ge_this": int((d[M["clo"]] >= r["clo_total"]).sum()),
                 "share": round(float((d[M["clo"]] >= r["clo_total"]).mean()), 4)})
cov = pd.DataFrame(rows); print("\nTraining-data coverage of manikin clo values:"); print(cov.to_string(index=False))
cov.to_csv(config.RESULTS / "clo_coverage.csv", index=False)
print("ASHRAE subset clo: max = %.2f, 95th pct = %.2f, 99th pct = %.2f" % (d[M["clo"]].max(), d[M["clo"]].quantile(.95), d[M["clo"]].quantile(.99)))

## 8. Exploratory figures (PNG 300 dpi + SVG in `figures/`)

In [ ]:
import explore
importlib.reload(explore)
explore.run_all(d, clo_tbl)
from IPython.display import Image, display
for f in sorted(config.FIGURES.glob("fig0*.png")):
    print(f.name); display(Image(filename=str(f), width=520))

---
# Stage 3: PMV baseline and ML models
Same inputs for every model: clo, air temperature, RH, air speed, met (PMV also uses mean radiant temperature, falling back to air temperature where missing). Validation is StratifiedGroupKFold by `building_id` (never random rows), plus leave-one-contributor-out as a robustness check. Intervals are cluster bootstrap over buildings.

In [ ]:
import models
importlib.reload(models)
df = models.set_target(models.load_clean(), "comfortable")   # ORIGINAL label (negative result)
pmv_chk = models.pmv_baseline(df)
print("rows:", len(df), "| buildings:", df[M["building_id"]].nunique(), "| rows with tr imputed from air temp:", pmv_chk.attrs["n_tr_imputed"])
print("ISO PMV rows with air speed > 1.0 m/s or air temp > 30 C (outside the ISO 7730 validity range):",
      int(((df[M["air_speed"]] > 1.0) | (df[M["air_temp"]] > 30)).sum()))
splits = models.building_splits(df)
print("Fold sizes (train, test):", [(len(a), len(b)) for a, b in splits])

## 10. Building-grouped 5-fold validation (main result)

In [ ]:
cls_b, sens_b, fold_b, oof_b, best_b = models.run_experiment(df, splits, "building 5-fold")
show_c = ["model", "accuracy", "macro_f1", "macro_f1_matched", "roc_auc", "roc_auc_ci95", "macro_f1_matched_ci95"]
print("COMFORT (vote in [-1,+1])"); print(cls_b[show_c].round(3).to_string(index=False))
print("\nSENSATION VOTE"); print(sens_b[["model", "mae", "mae_ci95", "within_1", "within_1_ci95"]].round(3).to_string(index=False))
print("\nChosen hyperparameters per fold (XGBoost classifier):", best_b["classifiers"]["XGBoost"])
print("Chosen hyperparameters per fold (quadratic logistic C):", [b["logisticregression__C"] for b in best_b["classifiers"]["Logistic regression (quadratic)"]])

## 11. Robustness: leave one contributor out (4 folds)

In [ ]:
csplits = models.contributor_splits(df)
print("Held-out contributors:", [(df[M["contributor"]].iloc[te[0]], len(te)) for _, te in csplits])
cls_c, sens_c, fold_c, oof_c, best_c = models.run_experiment(df, csplits, "leave-one-contributor-out")
print(cls_c[show_c].round(3).to_string(index=False))
print(sens_c[["model", "mae", "mae_ci95", "within_1", "within_1_ci95"]].round(3).to_string(index=False))
print("\nPer-fold (held-out contributor) ROC AUC:")
print(fold_c[fold_c.task == "comfort"].pivot(index="model", columns="fold", values="roc_auc").round(3).to_string())

## 12. Sensitivity: label = thermal acceptability (instead of sensation vote)

In [ ]:
acc = df[M["thermal_acceptability"]].map({"acceptable": 1, "unacceptable": 0})
print("Rows with acceptability:", int(acc.notna().sum()), "| share acceptable: %.3f" % acc.mean())
d_acc = df[acc.notna()].copy(); d_acc[models.Y] = acc[acc.notna()].astype(int).to_numpy()
cls_a, _, _, _, _ = models.run_experiment(d_acc, models.building_splits(d_acc), "acceptability label")
print(cls_a[show_c].round(3).to_string(index=False))

## 13. Figures and saved results

In [ ]:
for nm, t in (("neg_comfort_label_model_comparison", pd.concat([cls_b, cls_c, cls_a])), ("neg_comfort_label_sensation_comparison", pd.concat([sens_b, sens_c])),
              ("neg_comfort_label_fold_metrics", pd.concat([fold_b, fold_c]))):
    t.to_csv(config.RESULTS / f"{nm}.csv", index=False)
oof_b.to_csv(config.RESULTS / "neg_comfort_label_oof_predictions.csv.gz", index=False)
explore.fig_roc(oof_b); explore.fig_calibration(oof_b)
for f in ("fig07_roc_curves.png", "fig08_calibration.png"):
    print(f); display(Image(filename=str(config.FIGURES / f), width=480))

---\n# Stage 4: warm-side discomfort target, SHAP, scenarios, reports\nMain target = rounded sensation vote of +1 or more (decided after the stage 3 negative result). Runs the models against PMV, SHAP and permutation importance, the 29 degC / 85% RH scenarios with and without a fan (data-driven model and PMV side by side), then writes `results/summary.md` and `results/design_garment_budget.csv` from the saved numbers.

In [ ]:
import explain, scenarios, stage4, report
for m in (explain, scenarios, stage4, report): importlib.reload(m)
out = stage4.run()
print("Primary model:", out["primary_model"])
print(out["auc_by_validation"].round(3).to_string())

In [ ]:
show_c = ["validation", "model", "accuracy", "macro_f1", "macro_f1_matched", "roc_auc", "roc_auc_ci95"]
print(out["cls"][show_c].round(3).to_string(index=False))
print(out["sens"][["validation", "model", "mae", "mae_ci95", "within_1", "within_1_ci95"]].round(3).to_string(index=False))
print(out["shap"].round(3).to_string(index=False)); print(out["perm"].round(4).to_string(index=False)); print(out["coef"].round(3).to_string(index=False))
print("Fan levels:", out["fan"]); print("Highest comfortable total clo at 29 C / 85% RH:", out["limits_29_85"])
print("Bootstrap:", out["boot_29_85"])
print(out["limits_table"].round(2).to_string(index=False))

In [ ]:
report.build_all()
for f in ("fig09_shap_global_importance", "fig11_shap_dependence", "fig12_scenario_curves_29C_85RH", "fig13_scenario_heatmaps_temp_rh", "fig14_scenario_bedding_sets"):
    print(f); display(Image(filename=str(config.FIGURES / f"{f}.png"), width=620))
print(open(config.RESULTS / "summary.md").read()[:3000])

## Package everything

In [ ]:
import zipfile
out4 = PROJECT / "stage4_outputs.zip"
with zipfile.ZipFile(out4, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("results", "figures", "data/processed", "src", "notebooks"):
        for p in (PROJECT / sub).rglob("*"):
            if p.is_file() and "inspection" not in p.parts and "__pycache__" not in p.parts:
                z.write(p, p.relative_to(PROJECT))
    for name in ("README.md", "requirements.txt"):
        if (PROJECT / name).exists(): z.write(PROJECT / name, name)
print("Wrote", out4, f"({out4.stat().st_size/1024:.0f} kB)")
if IN_COLAB:
    files.download(str(out4))